> Note : (I think) You can not install the mysql inside the mysql inside the google colab because the same problem with docker may happen. The problem is, Google colab doesn't support deamons to run.



## Scenario 1

600+ Database + each having relationship with each other
Huge Database 1000s of Tables. If you have too many tables for the LLM context window, embed table descriptions (not database data):

In [ ]:
!pip install langchain langchain-community langchain-groq faiss-cpu sentence-transformers sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
# Setup Groq API Key
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"]=userdata.get("GROQ_API_KEY")

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain.embeddings.base import Embeddings
from langchain_core.documents import Document
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits.sql.base import create_sql_agent
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_classic.agents.agent_types import AgentType
import os
from typing import List

In [ ]:
# Create sample (Dummy Database)

!pip install -q faker


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 55.6 MB/s eta 0:00:00


In [ ]:
# COMPLETE SCRIPT TO CREATE SAMPLE DATABASE WITH TABLES AND DATA


import sqlite3
import random
from datetime import datetime, timedelta
from faker import Faker
import pandas as pd

# Install required packages
# !pip install faker pandas

# Initialize Faker for generating realistic data
fake = Faker()
Faker.seed(42)
random.seed(42)

# ============================================================================
# CREATE DATABASE CONNECTION
# ============================================================================
def create_database():
    """Create SQLite database connection"""
    conn = sqlite3.connect('company.db')
    cursor = conn.cursor()
    return conn, cursor

# ============================================================================
# TABLE 1: DEPARTMENTS
# ============================================================================
def create_departments_table(cursor):
    """Create and populate departments table"""
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS departments (
        department_id INTEGER PRIMARY KEY,
        department_name TEXT NOT NULL,
        manager_name TEXT,
        budget REAL,
        location TEXT,
        created_date DATE
    )
    ''')

    departments_data = [
        (1, 'Sales', 'John Smith', 500000.00, 'New York', '2020-01-15'),
        (2, 'Marketing', 'Sarah Johnson', 350000.00, 'Los Angeles', '2020-02-20'),
        (3, 'Engineering', 'Mike Chen', 800000.00, 'San Francisco', '2020-01-10'),
        (4, 'Human Resources', 'Emily Davis', 250000.00, 'Chicago', '2020-03-01'),
        (5, 'Finance', 'Robert Wilson', 400000.00, 'New York', '2020-01-20'),
        (6, 'Customer Support', 'Lisa Anderson', 300000.00, 'Austin', '2020-04-15'),
        (7, 'Product', 'David Lee', 600000.00, 'Seattle', '2020-02-10'),
        (8, 'Operations', 'Jennifer Brown', 450000.00, 'Boston', '2020-03-05'),
    ]

    cursor.executemany('''
    INSERT OR REPLACE INTO departments VALUES (?, ?, ?, ?, ?, ?)
    ''', departments_data)

    print("✓ Departments table created with 8 records")

# ============================================================================
# TABLE 2: EMPLOYEES
# ============================================================================
def create_employees_table(cursor):
    """Create and populate employees table"""
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS employees (
        employee_id INTEGER PRIMARY KEY,
        first_name TEXT NOT NULL,
        last_name TEXT NOT NULL,
        email TEXT UNIQUE,
        department_id INTEGER,
        position TEXT,
        salary REAL,
        hire_date DATE,
        performance_rating REAL,
        FOREIGN KEY (department_id) REFERENCES departments(department_id)
    )
    ''')

    positions = {
        1: ['Sales Rep', 'Sales Manager', 'Account Executive'],
        2: ['Marketing Specialist', 'Marketing Manager', 'Content Creator'],
        3: ['Software Engineer', 'Senior Engineer', 'Tech Lead', 'DevOps Engineer'],
        4: ['HR Specialist', 'Recruiter', 'HR Manager'],
        5: ['Accountant', 'Financial Analyst', 'Finance Manager'],
        6: ['Support Specialist', 'Support Manager', 'Technical Support'],
        7: ['Product Manager', 'Product Designer', 'Product Analyst'],
        8: ['Operations Manager', 'Operations Coordinator', 'Logistics Specialist'],
    }

    employees_data = []
    employee_id = 1

    for dept_id in range(1, 9):
        num_employees = random.randint(8, 15)
        for _ in range(num_employees):
            first_name = fake.first_name()
            last_name = fake.last_name()
            email = f"{first_name.lower()}.{last_name.lower()}@company.com"
            position = random.choice(positions[dept_id])
            salary = round(random.uniform(45000, 150000), 2)
            hire_date = fake.date_between(start_date='-5y', end_date='today')
            performance_rating = round(random.uniform(2.5, 5.0), 1)

            employees_data.append((
                employee_id, first_name, last_name, email, dept_id,
                position, salary, hire_date, performance_rating
            ))
            employee_id += 1

    cursor.executemany('''
    INSERT OR REPLACE INTO employees VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', employees_data)

    print(f"✓ Employees table created with {len(employees_data)} records")

# ============================================================================
# TABLE 3: PRODUCTS
# ============================================================================
def create_products_table(cursor):
    """Create and populate products table"""
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS products (
        product_id INTEGER PRIMARY KEY,
        product_name TEXT NOT NULL,
        category TEXT,
        price REAL,
        cost REAL,
        sku TEXT UNIQUE,
        stock_quantity INTEGER,
        created_date DATE
    )
    ''')

    categories = ['Electronics', 'Clothing', 'Home & Garden', 'Sports', 'Books', 'Toys']
    product_names = {
        'Electronics': ['Wireless Headphones', 'Smart Watch', 'Tablet', 'Bluetooth Speaker', 'Camera'],
        'Clothing': ['T-Shirt', 'Jeans', 'Jacket', 'Sneakers', 'Dress'],
        'Home & Garden': ['Coffee Maker', 'Blender', 'Vacuum Cleaner', 'Plant Pot', 'Lamp'],
        'Sports': ['Yoga Mat', 'Dumbbells', 'Running Shoes', 'Tennis Racket', 'Bicycle'],
        'Books': ['Fiction Novel', 'Cookbook', 'Self-Help Book', 'Biography', 'Technical Manual'],
        'Toys': ['Action Figure', 'Board Game', 'Puzzle', 'Doll', 'Building Blocks'],
    }

    products_data = []
    product_id = 1

    for category in categories:
        for product_name in product_names[category]:
            for variant in range(1, random.randint(2, 4)):
                cost = round(random.uniform(10, 200), 2)
                price = round(cost * random.uniform(1.5, 3.0), 2)
                sku = f"SKU-{category[:3].upper()}-{product_id:04d}"
                stock = random.randint(0, 500)
                created = fake.date_between(start_date='-2y', end_date='today')

                products_data.append((
                    product_id,
                    f"{product_name} - Variant {variant}",
                    category,
                    price,
                    cost,
                    sku,
                    stock,
                    created
                ))
                product_id += 1

    cursor.executemany('''
    INSERT OR REPLACE INTO products VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    ''', products_data)

    print(f"✓ Products table created with {len(products_data)} records")

# ============================================================================
# TABLE 4: ORDERS
# ============================================================================
def create_orders_table(cursor):
    """Create and populate orders table"""
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS orders (
        order_id INTEGER PRIMARY KEY,
        customer_name TEXT,
        customer_email TEXT,
        order_date DATE,
        ship_date DATE,
        delivery_date DATE,
        status TEXT,
        total_amount REAL,
        shipping_address TEXT,
        payment_method TEXT
    )
    ''')

    statuses = ['Pending', 'Shipped', 'Delivered', 'Cancelled', 'Processing']
    payment_methods = ['Credit Card', 'PayPal', 'Debit Card', 'Bank Transfer']

    orders_data = []

    for order_id in range(1, 1001):
        customer_name = fake.name()
        customer_email = fake.email()
        order_date = fake.date_between(start_date='-2y', end_date='today')
        ship_date = order_date + timedelta(days=random.randint(1, 3))
        delivery_date = ship_date + timedelta(days=random.randint(2, 7))
        status = random.choice(statuses)
        total_amount = round(random.uniform(20, 2000), 2)
        shipping_address = fake.address().replace('\n', ', ')
        payment_method = random.choice(payment_methods)

        orders_data.append((
            order_id, customer_name, customer_email, order_date, ship_date,
            delivery_date, status, total_amount, shipping_address, payment_method
        ))

    cursor.executemany('''
    INSERT OR REPLACE INTO orders VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', orders_data)

    print(f"✓ Orders table created with {len(orders_data)} records")

# ============================================================================
# TABLE 5: SALES_2023
# ============================================================================
def create_sales_2023_table(cursor):
    """Create and populate sales_2023 table"""
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS sales_2023 (
        sale_id INTEGER PRIMARY KEY,
        product_id INTEGER,
        order_id INTEGER,
        quantity INTEGER,
        unit_price REAL,
        total_price REAL,
        sale_date DATE,
        region TEXT,
        salesperson_id INTEGER,
        discount_percent REAL,
        FOREIGN KEY (product_id) REFERENCES products(product_id),
        FOREIGN KEY (order_id) REFERENCES orders(order_id)
    )
    ''')

    regions = ['North', 'South', 'East', 'West', 'Central']
    sales_data = []

    start_date = datetime(2023, 1, 1)
    end_date = datetime(2023, 12, 31)

    for sale_id in range(1, 5001):
        product_id = random.randint(1, 50)
        order_id = random.randint(1, 1000)
        quantity = random.randint(1, 20)
        unit_price = round(random.uniform(10, 500), 2)
        discount = round(random.uniform(0, 20), 1)
        total_price = round(quantity * unit_price * (1 - discount/100), 2)
        sale_date = fake.date_between(start_date=start_date, end_date=end_date)
        region = random.choice(regions)
        salesperson_id = random.randint(1, 30)

        sales_data.append((
            sale_id, product_id, order_id, quantity, unit_price,
            total_price, sale_date, region, salesperson_id, discount
        ))

    cursor.executemany('''
    INSERT OR REPLACE INTO sales_2023 VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', sales_data)

    print(f"✓ Sales_2023 table created with {len(sales_data)} records")

# ============================================================================
# TABLE 6: CUSTOMER_FEEDBACK
# ============================================================================
def create_customer_feedback_table(cursor):
    """Create and populate customer_feedback table"""
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS customer_feedback (
        feedback_id INTEGER PRIMARY KEY,
        customer_name TEXT,
        customer_email TEXT,
        product_id INTEGER,
        rating INTEGER,
        review_text TEXT,
        feedback_date DATE,
        ticket_status TEXT,
        category TEXT,
        FOREIGN KEY (product_id) REFERENCES products(product_id)
    )
    ''')

    ticket_statuses = ['Open', 'Closed', 'In Progress', 'Resolved']
    categories = ['Product Quality', 'Shipping', 'Customer Service', 'Pricing', 'General']

    reviews = [
        "Great product! Highly recommend.",
        "Not satisfied with the quality.",
        "Excellent customer service.",
        "Delivery was delayed.",
        "Good value for money.",
        "Product did not meet expectations.",
        "Fast shipping and great quality!",
        "Average product, nothing special.",
        "Will definitely buy again!",
        "Had some issues but customer service helped.",
    ]

    feedback_data = []

    for feedback_id in range(1, 2001):
        customer_name = fake.name()
        customer_email = fake.email()
        product_id = random.randint(1, 50)
        rating = random.randint(1, 5)
        review_text = random.choice(reviews)
        feedback_date = fake.date_between(start_date='-1y', end_date='today')
        ticket_status = random.choice(ticket_statuses)
        category = random.choice(categories)

        feedback_data.append((
            feedback_id, customer_name, customer_email, product_id,
            rating, review_text, feedback_date, ticket_status, category
        ))

    cursor.executemany('''
    INSERT OR REPLACE INTO customer_feedback VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', feedback_data)

    print(f"✓ Customer_feedback table created with {len(feedback_data)} records")

# ============================================================================
# TABLE 7: INVENTORY_2023
# ============================================================================
def create_inventory_2023_table(cursor):
    """Create and populate inventory_2023 table"""
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS inventory_2023 (
        inventory_id INTEGER PRIMARY KEY,
        product_id INTEGER,
        warehouse_location TEXT,
        quantity_in_stock INTEGER,
        reorder_level INTEGER,
        last_restock_date DATE,
        supplier_name TEXT,
        unit_cost REAL,
        FOREIGN KEY (product_id) REFERENCES products(product_id)
    )
    ''')

    warehouses = ['Warehouse A - NY', 'Warehouse B - CA', 'Warehouse C - TX',
                  'Warehouse D - FL', 'Warehouse E - IL']

    inventory_data = []

    for inventory_id in range(1, 501):
        product_id = random.randint(1, 50)
        warehouse = random.choice(warehouses)
        quantity = random.randint(0, 1000)
        reorder_level = random.randint(50, 200)
        last_restock = fake.date_between(start_date='-6m', end_date='today')
        supplier = fake.company()
        unit_cost = round(random.uniform(5, 150), 2)

        inventory_data.append((
            inventory_id, product_id, warehouse, quantity, reorder_level,
            last_restock, supplier, unit_cost
        ))

    cursor.executemany('''
    INSERT OR REPLACE INTO inventory_2023 VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    ''', inventory_data)

    print(f"✓ Inventory_2023 table created with {len(inventory_data)} records")

# ============================================================================
# MAIN FUNCTION TO CREATE ALL TABLES
# ============================================================================
def create_complete_database():
    """Create complete database with all tables and data"""
    print("\n" + "="*80)
    print("CREATING SAMPLE DATABASE: company.db")
    print("="*80 + "\n")

    conn, cursor = create_database()

    try:
        # Create all tables
        create_departments_table(cursor)
        create_employees_table(cursor)
        create_products_table(cursor)
        create_orders_table(cursor)
        create_sales_2023_table(cursor)
        create_customer_feedback_table(cursor)
        create_inventory_2023_table(cursor)

        # Commit changes
        conn.commit()

        print("\n" + "="*80)
        print("✓ DATABASE CREATED SUCCESSFULLY!")
        print("="*80 + "\n")

        # Display table statistics
        display_table_stats(cursor)

    except Exception as e:
        print(f"Error creating database: {e}")
        conn.rollback()
    finally:
        conn.close()

# ============================================================================
# DISPLAY TABLE STATISTICS
# ============================================================================
def display_table_stats(cursor):
    """Display statistics for all tables"""
    print("\nTABLE STATISTICS:")
    print("-" * 80)

    tables = ['departments', 'employees', 'products', 'orders',
              'sales_2023', 'customer_feedback', 'inventory_2023']

    for table in tables:
        cursor.execute(f"SELECT COUNT(*) FROM {table}")
        count = cursor.fetchone()[0]
        print(f"  {table.upper():<25} {count:>10} records")

    print("-" * 80)

# ============================================================================
# FUNCTION TO VERIFY DATA
# ============================================================================
def verify_database():
    """Verify database creation and display sample data"""
    conn = sqlite3.connect('company.db')

    print("\n" + "="*80)
    print("SAMPLE DATA FROM EACH TABLE")
    print("="*80 + "\n")

    # Sample from each table
    queries = {
        "Departments": "SELECT * FROM departments LIMIT 3",
        "Employees": "SELECT employee_id, first_name, last_name, position, salary FROM employees LIMIT 3",
        "Products": "SELECT product_id, product_name, category, price FROM products LIMIT 3",
        "Orders": "SELECT order_id, customer_name, order_date, status, total_amount FROM orders LIMIT 3",
        "Sales 2023": "SELECT sale_id, product_id, quantity, total_price, sale_date FROM sales_2023 LIMIT 3",
        "Customer Feedback": "SELECT feedback_id, customer_name, rating, category FROM customer_feedback LIMIT 3",
        "Inventory 2023": "SELECT inventory_id, product_id, warehouse_location, quantity_in_stock FROM inventory_2023 LIMIT 3"
    }

    for table_name, query in queries.items():
        print(f"\n{table_name}:")
        print("-" * 80)
        df = pd.read_sql_query(query, conn)
        print(df.to_string(index=False))

    conn.close()
    print("\n" + "="*80 + "\n")

# ============================================================================
# GET TABLE DESCRIPTIONS FOR VECTOR STORE
# ============================================================================
def get_table_descriptions():
    """Return table descriptions for vector store"""
    table_descriptions = [
        {
            "table": "employees",
            "description": "Employee information including salary, department, hire date, and performance reviews. Contains employee_id, first_name, last_name, email, department_id, position, salary, hire_date, performance_rating"
        },
        {
            "table": "sales_2023",
            "description": "Sales transactions from 2023 with product, customer, and revenue details. Contains sale_id, product_id, order_id, quantity, unit_price, total_price, sale_date, region, salesperson_id, discount_percent"
        },
        {
            "table": "customer_feedback",
            "description": "Customer support tickets and product reviews. Contains feedback_id, customer_name, customer_email, product_id, rating, review_text, feedback_date, ticket_status, category"
        },
        {
            "table": "products",
            "description": "Product catalog with names, categories, prices, and SKUs. Contains product_id, product_name, category, price, cost, sku, stock_quantity, created_date"
        },
        {
            "table": "inventory_2023",
            "description": "Stock levels and warehouse information for 2023. Contains inventory_id, product_id, warehouse_location, quantity_in_stock, reorder_level, last_restock_date, supplier_name, unit_cost"
        },
        {
            "table": "departments",
            "description": "Department information including managers and budgets. Contains department_id, department_name, manager_name, budget, location, created_date"
        },
        {
            "table": "orders",
            "description": "Customer orders with order dates, shipping info, and status. Contains order_id, customer_name, customer_email, order_date, ship_date, delivery_date, status, total_amount, shipping_address, payment_method"
        },
    ]

    return table_descriptions


# RUN THE SCRIPT

# Step 1: Create the database
create_complete_database()

# Step 2: Verify data was created
# Fire some sample query that will verify everythig is created as expected
verify_database()

# Step 3: Get table descriptions for your vector store
table_descriptions = get_table_descriptions()

print("\n✅ Database 'company.db' has been created successfully!")
print("✅ You can now use this database with the FAISS + Groq SQL agent code!")


# 7 tables with realistic data
# ~8,500 total recores
# sample queries to veriy everything is fine and well created
# Ready table description for later use


CREATING SAMPLE DATABASE: company.db

✓ Departments table created with 8 records
✓ Employees table created with 87 records
✓ Products table created with 59 records


/tmp/ipykernel_4452/2151400023.py:111: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.executemany('''
/tmp/ipykernel_4452/2151400023.py:169: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.executemany('''
/tmp/ipykernel_4452/2151400023.py:216: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.executemany('''


✓ Orders table created with 1000 records


/tmp/ipykernel_4452/2151400023.py:266: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.executemany('''


✓ Sales_2023 table created with 5000 records


/tmp/ipykernel_4452/2151400023.py:325: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.executemany('''


✓ Customer_feedback table created with 2000 records


/tmp/ipykernel_4452/2151400023.py:369: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.executemany('''


✓ Inventory_2023 table created with 500 records

✓ DATABASE CREATED SUCCESSFULLY!


TABLE STATISTICS:
--------------------------------------------------------------------------------
  DEPARTMENTS                        8 records
  EMPLOYEES                         87 records
  PRODUCTS                          59 records
  ORDERS                          1000 records
  SALES_2023                      5000 records
  CUSTOMER_FEEDBACK               2000 records
  INVENTORY_2023                   500 records
--------------------------------------------------------------------------------

SAMPLE DATA FROM EACH TABLE


Departments:
--------------------------------------------------------------------------------
 department_id department_name  manager_name   budget      location created_date
             1           Sales    John Smith 500000.0      New York   2020-01-15
             2       Marketing Sarah Johnson 350000.0   Los Angeles   2020-02-20
             3     Engineering     Mike

In [ ]:
# Groq Doesn't Provide Embeddings, so we use Huggingface embeddings ( free alternative )

from langchain_community.embeddings import HuggingFaceEmbeddings

# Initialize embeddings model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)


# table_descriptions = [
#     {
#         "table": "employees",
#         "description": "Employee information including salary, department, hire date, and performance reviews"
#     },
#     {
#         "table": "sales_2023",
#         "description": "Sales transactions from 2023 with product, customer, and revenue details"
#     },
#     {
#         "table": "customer_feedback",
#         "description": "Customer support tickets and product reviews"
#     },
#     {
#         "table": "products",
#         "description": "Product catalog with names, categories, prices, and SKUs"
#     },
#     {
#         "table": "inventory_2023",
#         "description": "Stock levels and warehouse information for 2023"
#     },
#     {
#         "table": "departments",
#         "description": "Department information including managers and budgets"
#     },
#     {
#         "table": "orders",
#         "description": "Customer orders with order dates, shipping info, and status"
#     },
#     # Add 493 more tables as needed...
# ]

/tmp/ipykernel_4452/1897443106.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Convert table description to Documents
docs = [
    Document(
        page_content=f"Table: {t['table']}\nDescription: {t['description']}",
        metadata={"table_name": t['table']}
    )
    for t in table_descriptions
]

# The Document() should contains fields such as "page_content"(Contains main information), and "metadata"(more information)


In [ ]:
# Create FAISS vectorstore from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

In [ ]:
# Optional: Save the vectorstore for later use ( in Local File System )
# vectorstore.save_local("table_descriptions_faiss_index")

# Optional: Load existing vectorstore
# vectorstore = FAISS.load_local(
#     "table_descriptions_faiss_index",
#     embeddings,
#     allow_dangerous_deserialization=True
# )

In [ ]:
# Function to find relevant tables based on the user query
def find_relevant_tables(question: str, k: int = 5) -> List[str]:
    """
    Find the most relevant tables for a given question using FAISS similarity search.

    Args:
        question: User's natural language question
        k: Number of relevant tables to return

    Returns:
        List of relevant table names
    """
    results = vectorstore.similarity_search(question, k=k)
    # "results" will be List of Document objects

    relevant_tables = [doc.metadata['table_name'] for doc in results]
    # Take table name only

    return relevant_tables


In [ ]:
# Example (Test it is perfectly works or not)
question = "What were our top selling products last year?"
relevant_tables = find_relevant_tables(question, k=5)
print(f"Question: {question}")
print(f"Relevant tables: {relevant_tables}")

Question: What were our top selling products last year?
Relevant tables: ['sales_2023', 'products', 'inventory_2023', 'customer_feedback', 'orders']


Out of all tables: **DEPARTMENTS,  EMPLOYEES,  PRODUCTS,  ORDERS,  SALES_2023,  CUSTOMER_FEEDBACK,  INVENTORY_2023**. We have fetched **'sales_2023', 'products', 'inventory_2023', 'customer_feedback', 'orders'**

In [ ]:
# Initialize Groq LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=1024
)

In [ ]:

# Create SQL connection with relevant_tables instead of ALL Tables.

# Connect to your database (SQLite example)
db = SQLDatabase.from_uri(
    "sqlite:///company.db",  # Replace with your database URI
    include_tables=relevant_tables,  # Only include relevant tables!
    sample_rows_in_table_info=2  # Reduce token usage
)

In [ ]:
# Create SQL toolkit ( Pass db to toolkit, Because the Agent accept tools not db object)
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

# That provides things like
# - list tables
# - read schema
# - execute SQL query

In [ ]:
# Create SQL agent
sql_agent = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=10,
    max_execution_time=60,
    early_stopping_method="generate"
)


This code is creating a smart AI agent that can talk to a SQL database using natural language(like English)

`create_sql_agent()` will create a "SQL Agent" and store it inside `sql_agent`

**What is the SQL Agent exactly internally?**
It is not magical thing. It is just a normal **LangChain Agent**(same as any other agent) but with **SQL-specific tools plugged into it**
- agent = The thinking brain(LLM + ReAct-style loop: Thought -> Action -> Observation -> Repeat)
- SQL toolkit = a set of 4-5 special tools that allows agent to talk with database
    - list out DBs
    - get scheam
    - execute query
    - (sometimes) check query looks safe

> `create_sql_agent` is just a convenience shortcut. Automatically builds the SQL toolkit for you.
- Pick a right prompt
- Wraps everything inside the `AgentExecutor`

<br>

The below is it's implementation from scratch for better understanding:
```
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import PromptTemplate

# 1. Connect to your database (same as before)
db = SQLDatabase.from_uri("your_database_uri_here")   # e.g. sqlite:///mydb.db

# 2. Create the SQL toolkit (this is the magic part)
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()          # ← these are the SQL tools

# 3. (Optional but recommended) Use a SQL-specific prompt
# You can also just use the default one from create_react_agent
SQL_PROMPT = """You are a helpful SQL expert.
You have access to a database. First list tables if needed,
then get schema, then write correct SQL, then run it.

{agent_scratchpad}"""

prompt = PromptTemplate.from_template(SQL_PROMPT)

# 4. Create the agent (this is what ZERO_SHOT_REACT_DESCRIPTION does)
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

# 5. Create the executor (exactly like your original code)
sql_agent = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=10,
    max_execution_time=60,
    early_stopping_method="generate"
)
```

1. `llm` : ChatModel / AI brain
2. `toolkit` : Give agent all tools it required
3. `agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION` : This is the thinking style of the agent. It means: "Think step by step, decide what to do, and describe your actions." It doesn't need examples — it figures things out on its own.
4. `verbose=True` : Show thinking steps
5. `max_iterations=10` : The agent can try maximum 10 times (think → act → think again) to solve your question. After 10 tries, it will stop.
- If it reaches 10 iterations without giving a final answer, it stops immediately with a message like: "Agent stopped due to iteration limit."
6. `max_execution_time=60` : Give the agent maximum 60 seconds to answer. If it takes longer, stop it (prevents it from running forever).
- This is a total time limit in seconds (here 60 seconds = 1 minute). It measures the entire running time from the moment you call sql_agent.run(...) or sql_agent.invoke(...).
- If the whole process (all thinking + all tool calls + all SQL executions) exceeds 60 seconds, it stops immediately, even if it has done only 2 or 3 iterations.
- It stops with a message like: "Agent stopped due to max execution time."

> Agent will stop as soon as ANY one of these two limits is hit: Either it completed 10 iterations OR Total time reached 60 seconds.  It is NOT that it gets exactly 10 tries spread across 60 seconds.

Example Scenario:
- If each iteration is fast (2–3 seconds), it may finish all 10 iterations in 20–30 seconds → stops because of iterations.
- If SQL queries are slow (e.g., complex joins), it may do only 4 iterations but take 65 seconds → stops because of time.
- If the agent finishes successfully in 3 iterations and 15 seconds → both limits are ignored, it returns the answer normally.

7. `early_stopping_method="generate"` : If the agent realizes it can't do better, stop early and just give the final answer instead of keep trying uselessly.

------------------------------------------------------

How `sql_agent` is different from the `create_react_agent` ?

|Feature              |create_sql_agent                        |create_react_agent             |
|---------------------|----------------------------------------|-------------------------------|
|Purpose              |Special shortcut only for SQL           |General-purpose for any tools  |
|Toolkit              |Automatically creates SQLDatabaseToolkit|You must give your own tools=  |
|Prompt               |Pre-made SQL-friendly prompt            |You must supply your own prompt|
|Agent Type           |Defaults to ZERO_SHOT_REACT_DESCRIPTION |Always ReAct style (you choose)|
|Extra features       |Handles db= directly, top_k rows, etc.  |None – fully manual            |
|Modern recommendation|Legacy (still works but not preferred)  |Recommended way now            |

> Simple way to remember: `create_sql_agent` = `create_react_agent` + pre-made SQL tools + SQL instructions.


> Types of Agents
|AgentType                       |Simple Meaning                                                               |
|--------------------------------|-----------------------------------------------------------------------------|
|ZERO_SHOT_REACT_DESCRIPTION     |Most common. Thinks step-by-step, no examples needed. (This is what you used)|
|REACT_DOCSTORE                  |ReAct + can look up info in a document store                                 |
|SELF_ASK_WITH_SEARCH            |Asks itself questions, then searches for answers                             |
|CONVERSATIONAL_REACT_DESCRIPTION|Remembers conversation history (good for chatbots)                           |

---

**What is Agent Toolkit?**

A toolkit is a collection of related tools bundled together.
Instead of giving your agent 10 separate tools one by one, you import a toolkit, and it gives you a ready group of tools (e.g., all SQL-related tools).

It contains toolkit for : SQLDatabase, FileManagement, Zapier, Jira, GitHub, Request(HTTP Requests), Json, SparkSQL, etc...

In [ ]:
# Complete Workflow Function

def query_database_with_rag(user_question: str, k_tables: int = 5):
    """
    Complete RAG workflow for querying database:
    1. Find relevant tables using FAISS
    2. Create SQL database connection with only relevant tables
    3. Use Groq-powered SQL agent to answer the question

    Args:
        user_question: User's natural language question
        k_tables: Number of relevant tables to consider

    Returns:
        Agent's response
    """
    print(f"\n{'='*80}")
    print(f"QUESTION: {user_question}")
    print(f"{'='*80}\n")

    # Step 1: Find relevant tables
    print("Step 1: Finding relevant tables...")
    relevant_tables = find_relevant_tables(user_question, k=k_tables)
    print(f"Relevant tables: {relevant_tables}\n")

    # Step 2: Create database connection with relevant tables
    print("Step 2: Connecting to database with relevant tables...")
    db = SQLDatabase.from_uri(
        "sqlite:///company.db",
        include_tables=relevant_tables,
        sample_rows_in_table_info=2
    )

    # Step 3: Create SQL agent
    print("Step 3: Creating SQL agent...")
    toolkit = SQLDatabaseToolkit(db=db, llm=llm)
    agent = create_sql_agent(
        llm=llm,
        toolkit=toolkit,
        agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
        verbose=True,
        handle_parsing_errors=True
    )

    # Step 4: Run the agent
    print("Step 4: Querying database...\n")
    response = agent.run(user_question)

    print(f"\n{'='*80}")
    print(f"ANSWER: {response}")
    print(f"{'='*80}\n")

    return response

In [ ]:
# EXAMPLE QUERIES
answer1 = query_database_with_rag(
    "What were our top selling products last year?"
)

answer1


QUESTION: What were our top selling products last year?

Step 1: Finding relevant tables...
Relevant tables: ['sales_2023', 'products', 'inventory_2023', 'customer_feedback', 'orders']

Step 2: Connecting to database with relevant tables...
Step 3: Creating SQL agent...
Step 4: Querying database...



> Entering new SQL Agent Executor chain...


/tmp/ipykernel_4452/3083848714.py:47: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  response = agent.run(user_question)


Thought: I should look at the tables in the database to see what I can query.  Then I should query the schema of the most relevant tables.
Action: sql_db_list_tables
Action Input: customer_feedback, inventory_2023, orders, products, sales_2023The observation from the sql_db_list_tables tool shows that there are several tables in the database, including "sales_2023" and "products". To find the top selling products, I will need to query the schema of these two tables to see what columns are available.

Action: sql_db_schema
Action Input: sales_2023, products
CREATE TABLE products (
	product_id INTEGER, 
	product_name TEXT NOT NULL, 
	category TEXT, 
	price REAL, 
	cost REAL, 
	sku TEXT, 
	stock_quantity INTEGER, 
	created_date DATE, 
	PRIMARY KEY (product_id), 
	UNIQUE (sku)
)

/*
2 rows from products table:
product_id	product_name	category	price	cost	sku	stock_quantity	created_date
1	Wireless Headphones - Variant 1	Electronics	270.56	137.25	SKU-ELE-0001	135	2026-01-02
2	Smart Watch - Va

'The top selling products last year were: \n1. Dress - Variant 1 (1315 units)\n2. Dumbbells - Variant 3 (1205 units)\n3. T-Shirt - Variant 2 (1202 units)\n4. T-Shirt - Variant 3 (1180 units)\n5. Wireless Headphones - Variant 1 (1177 units)\n6. Camera - Variant 2 (1173 units)\n7. Biography - Variant 2 (1158 units)\n8. Coffee Maker - Variant 2 (1153 units)\n9. Lamp - Variant 1 (1149 units)\n10. Smart Watch - Variant 1 (1144 units)'

In [ ]:
# ALTERNATIVE: Using create_sql_agent with custom prompt

In [ ]:
from langchain_classic.agents import AgentExecutor
from langchain_core.prompts import PromptTemplate


# Custom prompt for better SQL generation
SQL_PREFIX = """You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct SQL query to run, then look at the results of the query and return the answer.
You have access to the following tables: {table_names}

Only use the tables listed above. Do not query tables that are not in this list.
"""

ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (/usr/local/lib/python3.12/dist-packages/langchain/agents/__init__.py)

In [ ]:
def create_optimized_sql_agent(question: str, k_tables: int = 3):
    """
    Create an optimized SQL agent with custom configuration
    """
    # Find relevant tables
    relevant_tables = find_relevant_tables(question, k=k_tables)

    # Create database with relevant tables
    db = SQLDatabase.from_uri(
        "sqlite:///company.db",
        include_tables=relevant_tables,
        sample_rows_in_table_info=3
    )

    # Create toolkit and agent
    toolkit = SQLDatabaseToolkit(db=db, llm=llm)

    agent = create_sql_agent(
        llm=llm,
        toolkit=toolkit,
        agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
        verbose=True,
        prefix=SQL_PREFIX.format(table_names=", ".join(relevant_tables)),
        max_iterations=15,
        handle_parsing_errors=True
    )

    return agent, relevant_tables

In [ ]:
# Usage
agent, tables = create_optimized_sql_agent("What were the total sales in 2023?")
print(f"Using tables: {tables}")
result = agent.run("What were the total sales in 2023?")
print(result)

## Scenario 2

In [ ]:
!pip install langchain langchain-community langchain-groq langchain-huggingface faiss-cpu   sentence-transformers sqlalchemy langchainhub

In [ ]:
import sqlite3
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.tools import Tool
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_groq import ChatGroq
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import PromptTemplate, MessagesPlaceholder
from langchain_classic import hub
import os

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [ ]:
# Database setup
conn = sqlite3.connect("company.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    id INTEGER PRIMARY KEY,
    name TEXT,
    category TEXT,
    price REAL,
    description TEXT
)
""")

# Clear existing data and insert fresh data
cursor.execute("DELETE FROM products")
cursor.executemany("INSERT INTO products VALUES (?, ?, ?, ?, ?)", [
    (1, 'UltraBook Pro', 'Laptops', 1299.99, 'Lightweight laptop with 16GB RAM, perfect for developers and designers. Features long battery life and fast SSD storage.'),
    (2, 'GamerRig X', 'Laptops', 2499.99, 'High-performance gaming laptop with RTX 4080, RGB keyboard, and advanced cooling system for intense gaming sessions.'),
    (3, 'BudgetBook', 'Laptops', 499.99, 'Affordable laptop for students and basic office work. Ideal for web browsing and document editing.'),
    (4, 'WorkStation Elite', 'Laptops', 3299.99, 'Professional workstation for video editing and 3D rendering. Equipped with powerful CPU and 64GB RAM.'),
])
conn.commit()

# Extract descriptions and create embeddings
products_data = cursor.execute("SELECT id, name, description FROM products").fetchall()
conn.close()

In [ ]:
# Create documents for vector store
docs = [
    Document(
        page_content=f"{name}: {description}",
        metadata={"product_id": prod_id, "name": name}
    )
    for prod_id, name, description in products_data
]

In [ ]:
# Initialize HuggingFace embeddings (free alternative to OpenAI)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Create FAISS vector store
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

In [ ]:
# Optional: Save the FAISS index for later use
# vectorstore.save_local("faiss_product_index")

# Optional: Load existing FAISS index
# vectorstore = FAISS.load_local(
#     "faiss_product_index",
#     embeddings,
#     allow_dangerous_deserialization=True
# )

In [ ]:
# Initialize Groq LLM
# llm = ChatGroq(
#     model="llama-3.3-70b-versatile",  # or "mixtral-8x7b-32768", "gemma2-9b-it"
#     temperature=0,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
# )


# llm = ChatGroq(
#     model="llama-3.3-70b-versatile",
#     temperature=0,
#     max_tokens=4096,
#     max_retries=2,
# )


# Use a smaller, faster model to avoid rate limits
llm = ChatGroq(
    model="llama-3.1-8b-instant",  # Faster model with higher rate limits
    temperature=0,
    max_tokens=2048,
    max_retries=2,
)

In [ ]:
# SQL Database setup
db = SQLDatabase.from_uri("sqlite:///company.db")
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
sql_tools = toolkit.get_tools()

In [ ]:
# Semantic Search Tool
def semantic_product_search(query: str) -> str:
    """Search products by description content, features, or use cases"""
    try:
        results = vectorstore.similarity_search(query, k=3)
        output = []


        # "results" List of Document objects

        conn = sqlite3.connect("company.db")


        # Loop over "results"
        for doc in results:
            product = conn.execute(
                "SELECT * FROM products WHERE id = ?",
                (doc.metadata['product_id'],)
            ).fetchone()

            # product = One Row of Database


            if product:
                output.append(
                    f"Product: {product[1]}\n"
                    f"Category: {product[2]}\n"
                    f"Price: ${product[3]}\n"
                    f"Description: {product[4]}"
                )
        conn.close()

            ### Example Append ###
            # >>> l = []
            # >>> l.append("one")
            # >>> l
            # ['one']
            # >>> l.append("two""three")
            # >>> l
            # ['one', 'twothree']

        return "\n\n".join(output) if output else "No products found matching your query."
    except Exception as e:
        return f"Error during semantic search: {str(e)}"

semantic_tool = Tool(
    name="semantic_product_search",
    func=semantic_product_search,
    description=(
        "Use this to find products based on features, use cases, or descriptions. "
        "Good for queries like 'laptop for gaming', 'affordable device for students', "
        "'laptop for video editing', or 'lightweight laptop for developers'."
    )
)

In [ ]:
# Combine all tools
all_tools = sql_tools + [semantic_tool]

In [ ]:
# Create agent prompt
# prompt = ChatPromptTemplate.from_messages([
#     ("system", """You are a helpful product database assistant with access to both SQL queries and semantic search capabilities.

# You have two types of tools:

# 1. **SQL Tools** - Use these for:
#    - Price-based queries (e.g., "laptops under $1000", "average price")
#    - Category filtering and counting
#    - Exact matches and numerical comparisons
#    - Structured data queries

# 2. **Semantic Search Tool** - Use this for:
#    - Finding products by features or characteristics
#    - Use-case based searches (e.g., "for gaming", "for students", "for video editing")
#    - Descriptive queries about product capabilities
#    - Finding similar products

# **Guidelines:**
# - For price/numerical queries → Use SQL tools
# - For feature/use-case queries → Use semantic_product_search
# - For complex queries → Combine both approaches
# - Always provide clear, formatted responses with product details

# Examples:
# - "Show laptops under $1000" → Use SQL tools
# - "Find a laptop for video editing" → Use semantic_product_search
# - "Affordable laptops for students" → Use semantic_product_search, then optionally filter by price with SQL
# - "How many products in each category?" → Use SQL tools
# """),
#     ("human", "{input}"),
#     MessagesPlaceholder(variable_name="agent_scratchpad"),
# ])


# Not working with Groq models. Since Groq models doesn't accept the tool format as other LLM (OpenAI LLM) accept.











# Create ReAct prompt template
react_prompt = PromptTemplate.from_template("""You are a helpful product database assistant with access to both SQL queries and semantic search capabilities.

You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Guidelines:
- For price/numerical queries (e.g., "average price", "laptops under $1000") → Use sql_db_query
- For feature/use-case queries (e.g., "laptop for gaming", "for video editing") → Use semantic_product_search
- Always check the database schema first using sql_db_schema if needed
- Use sql_db_list_tables to see available tables

Begin!

Question: {input}
Thought: {agent_scratchpad}""")

`MessagesPlaceholder(variable_name="agent_scratchpad")` is the special empty slot in the prompt.

Think of the full prompt as a conversation template:
1. System Message (long instruction given to AI)
2. Human messages (user's question)
3. The placeholder <- empty slot
4. Later Ai will put things here


**Why it is needed ?**

This is used in AI Agents (like product assistant).

While answering your question, the AI may need to:
- Use tools ( SQL or semantic search )
- Get results from those tools
- Think step-by-step
- Try again if needed

All these intermediate thoughts and tool results are saved in the **"agent_scratchpad"**

> Imagine the AI is solving a math problem and it needs to do rough work on it's side. The main conversation is clean


------------------------------------------------



Q. When do you need to add `MessagesPlaceholder(variable_name="agent_scratchpad")`?

You need it only when you are building an "Agent" — that is, an AI that can use tools (like SQL tool, semantic search tool, calculator, etc.) and may need to take multiple steps to answer one question.

Do NOT use scratchpad → When it's a simple chatbot or single-step LLM call (no tools involved)


-----------------------------------------------------

Common misconception: If you use the scratchpad then the thinking of the AI will hide. and you will not see the thinking of the AI, only see the clear response.

With scratchpad (Agent setup):
- AI can think step-by-step
- It can call tools multiple times
- Alll thinking, tool calls, and tool results are stored in scratchpad
- After finishing, the AI gives a final clean answer.
- But ... the messy thinking is usually still shown to user unless you hide it in your code ( Using some custom logic)

In [ ]:
# Create agent (using create_tool_calling_agent for Groq compatibility)
# agent = create_tool_calling_agent(
#     llm=llm,
#     tools=all_tools,
#     prompt=react_prompt
# )

In [ ]:
# Create agent executor
# agent_executor = AgentExecutor(
#     agent=agent,
#     tools=all_tools,
#     verbose=True,
#     handle_parsing_errors=True,
#     max_iterations=5
# )


# agent_executor = AgentExecutor(
#     agent=agent,
#     tools=all_tools,
#     verbose=True,
#     handle_parsing_errors=True,
#     max_iterations=10,
#     return_intermediate_steps=True
# )

In [ ]:
# result1 = agent_executor.invoke({"input": "What's the average price of laptops?"})
# print(f"\nResult: {result1['output']}")

In [ ]:
from langchain_classic.agents import initialize_agent, AgentType

agent_executor = initialize_agent(
    tools=all_tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10,
    early_stopping_method="generate"
)

In [ ]:
result1 = agent_executor.invoke({"input": "What's the average price of laptops?"})
print(f"\nFinal Answer: {result1['output']}")



> Entering new AgentExecutor chain...
Thought: To find the average price of laptops, I need to query the database for the price of laptops. I should first check if the 'price' column exists in the 'laptops' table.

Action: sql_db_schema
Action Input: laptops
Observation: Error: table_names {'laptops'} not found in database
Thought:Thought: It seems that the 'laptops' table does not exist in the database. I should first list all the tables in the database to see if there's a similar table that I can query.

Action: sql_db_list_tables
Action Input: 
Observation: products
Thought:Question: What's the average price of laptops?
Thought: It seems that the 'products' table exists in the database, which might contain information about laptops. I should check the schema of the 'products' table to see if it has a 'price' column.

Action: sql_db_schema
Action Input: products
Observation: 
CREATE TABLE products (
	id INTEGER, 
	name TEXT, 
	category TEXT, 
	price REAL, 
	description TEXT, 
	PRIM

In [ ]:
result2 = agent_executor.invoke({"input": "I need a laptop for video editing and 3D work"})
print(f"\nFinal Answer: {result2['output']}")



> Entering new AgentExecutor chain...
Thought: To find a suitable laptop for video editing and 3D work, I need to consider the specifications and features of the laptop. I should look for laptops with powerful processors, high-end graphics cards, and sufficient RAM.

Action: semantic_product_search
Action Input: 'laptop for video editing and 3D work'
Observation: Product: WorkStation Elite
Category: Laptops
Price: $3299.99
Description: Professional workstation for video editing and 3D rendering. Equipped with powerful CPU and 64GB RAM.

Product: BudgetBook
Category: Laptops
Price: $499.99
Description: Affordable laptop for students and basic office work. Ideal for web browsing and document editing.

Product: UltraBook Pro
Category: Laptops
Price: $1299.99
Description: Lightweight laptop with 16GB RAM, perfect for developers and designers. Features long battery life and fast SSD storage.
Thought:Thought: The semantic product search tool has provided me with three potential laptops for

In [ ]:
result3 = agent_executor.invoke({"input": "Find affordable laptops good for students"})
print(f"\nFinal Answer: {result3['output']}")



> Entering new AgentExecutor chain...
Thought: To find affordable laptops good for students, I need to search for products that match this description.
Action: semantic_product_search
Action Input: 'affordable laptop for students'
Observation: Product: BudgetBook
Category: Laptops
Price: $499.99
Description: Affordable laptop for students and basic office work. Ideal for web browsing and document editing.

Product: UltraBook Pro
Category: Laptops
Price: $1299.99
Description: Lightweight laptop with 16GB RAM, perfect for developers and designers. Features long battery life and fast SSD storage.

Product: WorkStation Elite
Category: Laptops
Price: $3299.99
Description: Professional workstation for video editing and 3D rendering. Equipped with powerful CPU and 64GB RAM.
Thought:Thought: The product search returned three laptops, but I need to find the affordable ones.
Action: semantic_product_search
Action Input: 'affordable laptop'
Observation: Product: BudgetBook
Category: Laptops
Pri

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kev2426xet8vma443nmzk0k7` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 4140, Requested 1904. Please try again in 440ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
# Another way
# Solution 2: Direct Function Calling ( No Agent Loop )

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def intelligent_product_assistant(user_query: str) -> str:
    """Direct assistant without complex agent loops"""

    # Step 1: Classify query type
    classifier_prompt = ChatPromptTemplate.from_messages([
        ("system", "Classify the query as 'SQL' for price/counting queries or 'SEMANTIC' for feature/use-case searches. Reply with just one word."),
        ("human", "{query}")
    ])

    llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
    classifier = classifier_prompt | llm | StrOutputParser()
    query_type = classifier.invoke({"query": user_query}).strip().upper()

    # Step 2: Execute appropriate search
    if "SEMANTIC" in query_type:
        search_results = semantic_product_search(user_query)
        context = f"Semantic search results:\n{search_results}"
    else:
        # Generate and execute SQL
        sql_prompt = ChatPromptTemplate.from_messages([
            ("system", f"Generate a SQLite query for this question. Database schema:\n{db.get_table_info()}\n\nReturn ONLY the SQL query."),
            ("human", "{query}")
        ])
        sql_generator = sql_prompt | llm | StrOutputParser()
        sql_query = sql_generator.invoke({"query": user_query})
        sql_query = sql_query.replace("```sql", "").replace("```", "").strip()

        try:
            sql_results = db.run(sql_query)
            context = f"SQL query: {sql_query}\nResults: {sql_results}"
        except Exception as e:
            context = f"Error executing SQL: {e}"

    # Step 3: Generate final response
    response_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful product assistant. Based on the search results, provide a clear, concise answer."),
        ("human", "User question: {query}\n\n{context}\n\nProvide a helpful answer:")
    ])

    responder = response_prompt | llm | StrOutputParser()
    return responder.invoke({"query": user_query, "context": context})




In [ ]:
# Test
print("\n=== Semantic Search ===")
print(intelligent_product_assistant("I need a laptop for video editing and 3D work"))


=== Semantic Search ===
For demanding video‑editing and 3D‑rendering tasks you’ll want a machine with a high‑performance CPU, plenty of RAM, and a professional‑grade GPU.  

**Best match from the list:** **WorkStation Elite**  
- **Category:** Laptop (Professional workstation)  
- **Price:** **$3,299.99**  
- **Key specs:** Powerful CPU (unspecified but marketed for workstation use) + **64 GB RAM** – ideal for handling large video files and complex 3D scenes.  
- **Why it fits:** The combination of a high‑end processor and a large memory pool gives you the headroom needed for smooth timeline playback, fast rendering, and multitasking with editing software (e.g., Adobe Premiere, DaVinci Resolve, Blender, Maya).  

**Other options to consider (if budget is tighter):**  

| Laptop | Price | Specs | Suitability |
|--------|-------|-------|--------------|
| UltraBook Pro | $1,299.99 | 16 GB RAM, fast SSD, lightweight | Good for light‑to‑moderate editing and 3D work, but you may need to upg

In [ ]:
print("\n=== Hybrid Query ===")
print(intelligent_product_assistant("Find affordable laptops good for students"))


=== Hybrid Query ===
**Recommended Affordable Laptop for Students**

| Product | Price | Why It’s a Good Fit for Students |
|---------|-------|-----------------------------------|
| **BudgetBook** | **$499.99** | • Designed specifically for students and basic office work<br>• Handles web browsing, document editing, video streaming, and light multitasking with ease<br>• Lightweight and portable for campus life<br>• Budget‑friendly without sacrificing essential performance |

**Key Benefits**

- **Cost‑Effective:** At just under $500, it fits most student budgets and leaves room for accessories or software.
- **Student‑Centric Features:** The hardware is optimized for everyday tasks like research, online classes, and assignments.
- **Portability:** Easy to carry between classes, libraries, and coffee shops.
- **Battery Life:** Sufficient for a full day of classes (typical for budget laptops).

If you need a bit more power for specialized tasks (e.g., graphic design, programming), you mi

In [ ]:
# Solution 3
# Simple If-Else Router (Most Reliable)

In [ ]:
def simple_product_search(user_query: str) -> str:
    """Simple keyword-based routing"""

    query_lower = user_query.lower()

    # SQL keywords
    sql_keywords = ['average', 'count', 'how many', 'total', 'sum', 'price of all', 'list all']

    # Check if it's a SQL query
    is_sql = any(keyword in query_lower for keyword in sql_keywords)

    if is_sql:
        # Use SQL
        llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
        sql_prompt = ChatPromptTemplate.from_messages([
            ("system", f"Generate SQLite query. Schema:\n{db.get_table_info()}\nReturn only SQL."),
            ("human", "{query}")
        ])
        sql_gen = sql_prompt | llm | StrOutputParser()
        sql = sql_gen.invoke({"query": user_query}).replace("```sql", "").replace("```", "").strip()
        result = db.run(sql)
        return f"Query: {sql}\n\nResult: {result}"
    else:
        # Use semantic search
        return semantic_product_search(user_query)

In [ ]:
print(simple_product_search("I need a laptop for video editing and 3D work"))

Product: WorkStation Elite
Category: Laptops
Price: $3299.99
Description: Professional workstation for video editing and 3D rendering. Equipped with powerful CPU and 64GB RAM.

Product: BudgetBook
Category: Laptops
Price: $499.99
Description: Affordable laptop for students and basic office work. Ideal for web browsing and document editing.

Product: UltraBook Pro
Category: Laptops
Price: $1299.99
Description: Lightweight laptop with 16GB RAM, perfect for developers and designers. Features long battery life and fast SSD storage.


In [ ]:
print(simple_product_search("Find affordable laptops good for students"))

Product: BudgetBook
Category: Laptops
Price: $499.99
Description: Affordable laptop for students and basic office work. Ideal for web browsing and document editing.

Product: UltraBook Pro
Category: Laptops
Price: $1299.99
Description: Lightweight laptop with 16GB RAM, perfect for developers and designers. Features long battery life and fast SSD storage.

Product: WorkStation Elite
Category: Laptops
Price: $3299.99
Description: Professional workstation for video editing and 3D rendering. Equipped with powerful CPU and 64GB RAM.


# LangGraph Fusion Approch : SQL Agent + Semantic Search

In [ ]:
# Not working

In [ ]:
# Again try the Fusion

In [ ]:
# ==============
# FUSION APPROACH: SQL AGENT + SEMANTIC SEARCH
# ==============
# When no embedding needed → SQL agent works
# When embedding needed → Fetch similar documents from vector database
# ==============

!pip install -qq langchain langchain-community langchain-groq langchain-huggingface faiss-cpu sentence-transformers sqlalchemy faker pandas langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================

import os
import sqlite3
import random
from datetime import datetime, timedelta
from faker import Faker
import pandas as pd
from typing import List, Dict, Tuple, Optional

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.tools import Tool
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_groq import ChatGroq
from langchain_classic.agents import initialize_agent, AgentType, create_sql_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Setup API Key
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================

import os
import sqlite3
import random
from datetime import datetime, timedelta
from faker import Faker
import pandas as pd
from typing import List, Dict, Tuple, Optional

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.tools import Tool
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_groq import ChatGroq
from langchain_classic.agents import initialize_agent, AgentType, create_sql_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Setup API Key
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [ ]:
# ============================================================================
# SECTION 1: DATABASE CREATION (from Scenario 1)
# ============================================================================

# Initialize Faker for generating realistic data
fake = Faker()
Faker.seed(42)
random.seed(42)

def create_database():  # "company.db"
    """Create SQLite database connection"""
    conn = sqlite3.connect('company.db')
    cursor = conn.cursor()
    return conn, cursor

def create_departments_table(cursor):
    """Create and populate departments table"""
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS departments (
        department_id INTEGER PRIMARY KEY,
        department_name TEXT NOT NULL,
        manager_name TEXT,
        budget REAL,
        location TEXT,
        created_date DATE
    )
    ''')

    departments_data = [
        (1, 'Sales', 'John Smith', 500000.00, 'New York', '2020-01-15'),
        (2, 'Marketing', 'Sarah Johnson', 350000.00, 'Los Angeles', '2020-02-20'),
        (3, 'Engineering', 'Mike Chen', 800000.00, 'San Francisco', '2020-01-10'),
        (4, 'Human Resources', 'Emily Davis', 250000.00, 'Chicago', '2020-03-01'),
        (5, 'Finance', 'Robert Wilson', 400000.00, 'New York', '2020-01-20'),
        (6, 'Customer Support', 'Lisa Anderson', 300000.00, 'Austin', '2020-04-15'),
        (7, 'Product', 'David Lee', 600000.00, 'Seattle', '2020-02-10'),
        (8, 'Operations', 'Jennifer Brown', 450000.00, 'Boston', '2020-03-05'),
    ]

    cursor.executemany('''
    INSERT OR REPLACE INTO departments VALUES (?, ?, ?, ?, ?, ?)
    ''', departments_data)

    print("✓ Departments table created with 8 records")

def create_employees_table(cursor):
    """Create and populate employees table"""
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS employees (
        employee_id INTEGER PRIMARY KEY,
        first_name TEXT NOT NULL,
        last_name TEXT NOT NULL,
        email TEXT UNIQUE,
        department_id INTEGER,
        position TEXT,
        salary REAL,
        hire_date DATE,
        performance_rating REAL,
        FOREIGN KEY (department_id) REFERENCES departments(department_id)
    )
    ''')

    positions = {
        1: ['Sales Rep', 'Sales Manager', 'Account Executive'],
        2: ['Marketing Specialist', 'Marketing Manager', 'Content Creator'],
        3: ['Software Engineer', 'Senior Engineer', 'Tech Lead', 'DevOps Engineer'],
        4: ['HR Specialist', 'Recruiter', 'HR Manager'],
        5: ['Accountant', 'Financial Analyst', 'Finance Manager'],
        6: ['Support Specialist', 'Support Manager', 'Technical Support'],
        7: ['Product Manager', 'Product Designer', 'Product Analyst'],
        8: ['Operations Manager', 'Operations Coordinator', 'Logistics Specialist'],
    }

    employees_data = []
    employee_id = 1

    for dept_id in range(1, 9):
        num_employees = random.randint(8, 15)
        for _ in range(num_employees):
            first_name = fake.first_name()
            last_name = fake.last_name()
            email = f"{first_name.lower()}.{last_name.lower()}{employee_id}@company.com"
            position = random.choice(positions[dept_id])
            salary = round(random.uniform(45000, 150000), 2)
            hire_date = fake.date_between(start_date='-5y', end_date='today')
            performance_rating = round(random.uniform(2.5, 5.0), 1)

            employees_data.append((
                employee_id, first_name, last_name, email, dept_id,
                position, salary, hire_date, performance_rating
            ))
            employee_id += 1

    cursor.executemany('''
    INSERT OR REPLACE INTO employees VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', employees_data)

    print(f"✓ Employees table created with {len(employees_data)} records")

def create_products_table(cursor):
    """Create and populate products table with detailed descriptions"""
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS products (
        product_id INTEGER PRIMARY KEY,
        product_name TEXT NOT NULL,
        category TEXT,
        price REAL,
        cost REAL,
        sku TEXT UNIQUE,
        stock_quantity INTEGER,
        description TEXT,
        features TEXT,
        created_date DATE
    )
    ''')

    # Product data with descriptions and features for semantic search
    products_with_descriptions = [
        # Electronics
        (1, 'Wireless Headphones Pro', 'Electronics', 299.99, 120.00, 'SKU-ELE-0001', 150,
         'Premium wireless headphones with active noise cancellation, perfect for music lovers and professionals who need to focus in noisy environments.',
         'ANC, 30hr battery, Bluetooth 5.0, comfortable ear cushions'),
        (2, 'Smart Watch Elite', 'Electronics', 449.99, 200.00, 'SKU-ELE-0002', 80,
         'Advanced smartwatch for fitness enthusiasts and health-conscious users. Tracks heart rate, sleep, and over 100 workout types.',
         'Heart rate monitor, GPS, waterproof, 7-day battery'),
        (3, 'Bluetooth Speaker Max', 'Electronics', 179.99, 70.00, 'SKU-ELE-0003', 200,
         'Powerful portable speaker ideal for outdoor parties and adventures. Waterproof design for beach and pool use.',
         '360-degree sound, waterproof IPX7, 20hr playtime'),
        (4, 'USB-C Hub Professional', 'Electronics', 89.99, 35.00, 'SKU-ELE-0004', 300,
         'Essential hub for remote workers and digital nomads. Connect multiple devices to your laptop seamlessly.',
         '7-in-1, HDMI 4K, USB 3.0, SD card reader'),
        (5, 'Wireless Charging Pad', 'Electronics', 49.99, 15.00, 'SKU-ELE-0005', 500,
         'Fast wireless charger compatible with all Qi-enabled devices. Sleek design for home or office.',
         '15W fast charging, LED indicator, anti-slip surface'),

        # Clothing
        (6, 'Performance T-Shirt', 'Clothing', 45.99, 15.00, 'SKU-CLO-0006', 400,
         'Moisture-wicking athletic shirt perfect for gym workouts and running. Keeps you cool and dry.',
         'Quick-dry fabric, breathable, anti-odor technology'),
        (7, 'Classic Denim Jeans', 'Clothing', 79.99, 30.00, 'SKU-CLO-0007', 250,
         'Timeless denim jeans suitable for casual office wear and weekend outings. Comfortable stretch fit.',
         'Premium cotton blend, stretch comfort, classic fit'),
        (8, 'Winter Jacket Extreme', 'Clothing', 199.99, 80.00, 'SKU-CLO-0008', 100,
         'Heavy-duty winter jacket for extreme cold weather. Ideal for skiing, hiking, and outdoor winter activities.',
         'Down insulation, waterproof, windproof, -30°C rated'),
        (9, 'Running Shoes Ultra', 'Clothing', 149.99, 60.00, 'SKU-CLO-0009', 180,
         'Lightweight running shoes designed for marathon runners and daily joggers. Superior cushioning and support.',
         'Carbon fiber plate, responsive foam, breathable mesh'),
        (10, 'Business Casual Dress', 'Clothing', 129.99, 50.00, 'SKU-CLO-0010', 120,
         'Elegant dress appropriate for business meetings and formal events. Wrinkle-resistant fabric.',
         'Wrinkle-free, machine washable, versatile style'),

        # Home & Garden
        (11, 'Smart Coffee Maker', 'Home & Garden', 249.99, 100.00, 'SKU-HOM-0011', 90,
         'WiFi-enabled coffee maker for busy professionals. Schedule your morning coffee from bed using the app.',
         'App control, programmable, 12-cup capacity, auto-clean'),
        (12, 'Robot Vacuum Pro', 'Home & Garden', 599.99, 250.00, 'SKU-HOM-0012', 60,
         'Autonomous vacuum cleaner perfect for pet owners and busy families. Handles pet hair with ease.',
         'LIDAR navigation, pet hair pickup, self-emptying, app control'),
        (13, 'Air Purifier Max', 'Home & Garden', 349.99, 140.00, 'SKU-HOM-0013', 75,
         'Hospital-grade air purifier for allergy sufferers and health-conscious families. Removes 99.97% of allergens.',
         'HEPA filter, covers 500 sq ft, quiet operation, air quality sensor'),
        (14, 'Smart LED Bulbs Set', 'Home & Garden', 59.99, 20.00, 'SKU-HOM-0014', 400,
         'Color-changing smart bulbs for home automation enthusiasts. Create perfect ambiance for any mood.',
         '16 million colors, voice control, scheduling, energy efficient'),
        (15, 'Indoor Herb Garden Kit', 'Home & Garden', 129.99, 50.00, 'SKU-HOM-0015', 150,
         'Hydroponic herb garden for cooking enthusiasts and urban dwellers. Grow fresh herbs year-round.',
         'LED grow lights, self-watering, includes seed pods'),

        # Sports
        (16, 'Premium Yoga Mat', 'Sports', 89.99, 30.00, 'SKU-SPO-0016', 200,
         'Extra-thick yoga mat for beginners and advanced yogis. Non-slip surface for all yoga styles.',
         '6mm thickness, non-slip, eco-friendly TPE, carrying strap'),
        (17, 'Adjustable Dumbbell Set', 'Sports', 399.99, 180.00, 'SKU-SPO-0017', 50,
         'Space-saving dumbbells for home gym enthusiasts. Replaces 15 pairs of dumbbells.',
         '5-52.5 lbs adjustable, quick change, compact storage'),
        (18, 'Fitness Tracker Band', 'Sports', 79.99, 25.00, 'SKU-SPO-0018', 300,
         'Affordable fitness tracker for beginners starting their health journey. Easy to use interface.',
         'Step counter, sleep tracking, heart rate, 7-day battery'),
        (19, 'Professional Tennis Racket', 'Sports', 249.99, 100.00, 'SKU-SPO-0019', 80,
         'Tournament-grade racket for competitive players and serious tennis enthusiasts.',
         'Carbon fiber frame, optimal balance, vibration dampening'),
        (20, 'Mountain Bike Explorer', 'Sports', 899.99, 400.00, 'SKU-SPO-0020', 30,
         'All-terrain mountain bike for adventure seekers and trail riders. Handles rough terrain with ease.',
         '21-speed, hydraulic brakes, front suspension, aluminum frame'),
    ]

    cursor.executemany('''
    INSERT OR REPLACE INTO products VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, date('now'))
    ''', products_with_descriptions)

    print(f"✓ Products table created with {len(products_with_descriptions)} records (with descriptions)")

def create_orders_table(cursor):
    """Create and populate orders table"""
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS orders (
        order_id INTEGER PRIMARY KEY,
        customer_name TEXT,
        customer_email TEXT,
        order_date DATE,
        ship_date DATE,
        delivery_date DATE,
        status TEXT,
        total_amount REAL,
        shipping_address TEXT,
        payment_method TEXT
    )
    ''')

    statuses = ['Pending', 'Shipped', 'Delivered', 'Cancelled', 'Processing']
    payment_methods = ['Credit Card', 'PayPal', 'Debit Card', 'Bank Transfer']

    orders_data = []

    for order_id in range(1, 501):
        customer_name = fake.name()
        customer_email = fake.email()
        order_date = fake.date_between(start_date='-1y', end_date='today')
        ship_date = order_date + timedelta(days=random.randint(1, 3))
        delivery_date = ship_date + timedelta(days=random.randint(2, 7))
        status = random.choice(statuses)
        total_amount = round(random.uniform(20, 2000), 2)
        shipping_address = fake.address().replace('\n', ', ')
        payment_method = random.choice(payment_methods)

        orders_data.append((
            order_id, customer_name, customer_email, order_date, ship_date,
            delivery_date, status, total_amount, shipping_address, payment_method
        ))

    cursor.executemany('''
    INSERT OR REPLACE INTO orders VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', orders_data)

    print(f"✓ Orders table created with {len(orders_data)} records")

def create_sales_2023_table(cursor):
    """Create and populate sales_2023 table"""
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS sales_2023 (
        sale_id INTEGER PRIMARY KEY,
        product_id INTEGER,
        order_id INTEGER,
        quantity INTEGER,
        unit_price REAL,
        total_price REAL,
        sale_date DATE,
        region TEXT,
        salesperson_id INTEGER,
        discount_percent REAL,
        FOREIGN KEY (product_id) REFERENCES products(product_id),
        FOREIGN KEY (order_id) REFERENCES orders(order_id)
    )
    ''')

    regions = ['North', 'South', 'East', 'West', 'Central']
    sales_data = []

    start_date = datetime(2023, 1, 1)
    end_date = datetime(2023, 12, 31)

    for sale_id in range(1, 2001):
        product_id = random.randint(1, 20)
        order_id = random.randint(1, 500)
        quantity = random.randint(1, 10)
        unit_price = round(random.uniform(30, 500), 2)
        discount = round(random.uniform(0, 20), 1)
        total_price = round(quantity * unit_price * (1 - discount/100), 2)
        sale_date = fake.date_between(start_date=start_date, end_date=end_date)
        region = random.choice(regions)
        salesperson_id = random.randint(1, 30)

        sales_data.append((
            sale_id, product_id, order_id, quantity, unit_price,
            total_price, sale_date, region, salesperson_id, discount
        ))

    cursor.executemany('''
    INSERT OR REPLACE INTO sales_2023 VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', sales_data)

    print(f"✓ Sales_2023 table created with {len(sales_data)} records")

def create_customer_feedback_table(cursor):
    """Create and populate customer_feedback table with detailed reviews"""
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS customer_feedback (
        feedback_id INTEGER PRIMARY KEY,
        customer_name TEXT,
        customer_email TEXT,
        product_id INTEGER,
        rating INTEGER,
        review_text TEXT,
        feedback_date DATE,
        ticket_status TEXT,
        category TEXT,
        FOREIGN KEY (product_id) REFERENCES products(product_id)
    )
    ''')

    ticket_statuses = ['Open', 'Closed', 'In Progress', 'Resolved']
    categories = ['Product Quality', 'Shipping', 'Customer Service', 'Pricing', 'General']

    # Detailed reviews for semantic search
    reviews_positive = [
        "Absolutely love this product! Exceeded all my expectations. The quality is outstanding and it arrived quickly.",
        "Best purchase I've made this year. Works perfectly and looks great. Highly recommend to everyone!",
        "Great value for money. The features are exactly what I needed. Customer service was also very helpful.",
        "Five stars! The product is durable, well-designed, and easy to use. Will definitely buy again.",
        "Impressed with the quality. It's perfect for my needs and the delivery was super fast.",
        "This product has changed my daily routine for the better. Couldn't be happier with my purchase!",
        "Excellent build quality and the performance is amazing. Worth every penny spent.",
        "Really satisfied with this purchase. It works as advertised and the packaging was great.",
    ]

    reviews_negative = [
        "Disappointed with the quality. Product broke after just two weeks of normal use. Not worth the price.",
        "Shipping took forever and the product arrived damaged. Had to go through a lengthy return process.",
        "Does not work as advertised. Very frustrating experience. Would not recommend.",
        "Poor customer service experience. They took ages to respond and didn't resolve my issue.",
        "The product looks cheap and feels flimsy. Expected much better quality for this price point.",
        "Had multiple issues with this product. It stopped working after a month. Very disappointed.",
    ]

    reviews_neutral = [
        "It's okay. Does what it's supposed to do but nothing special. Average quality.",
        "Decent product for the price. Has some minor issues but overall acceptable.",
        "Works fine but the design could be better. Not bad, not great either.",
        "Met my basic expectations. Wouldn't say I'm thrilled but it gets the job done.",
    ]

    feedback_data = []

    for feedback_id in range(1, 801):
        customer_name = fake.name()
        customer_email = fake.email()
        product_id = random.randint(1, 20)
        rating = random.choices([1, 2, 3, 4, 5], weights=[5, 10, 15, 30, 40])[0]

        # Select review based on rating
        if rating >= 4:
            review_text = random.choice(reviews_positive)
        elif rating <= 2:
            review_text = random.choice(reviews_negative)
        else:
            review_text = random.choice(reviews_neutral)

        feedback_date = fake.date_between(start_date='-1y', end_date='today')
        ticket_status = random.choice(ticket_statuses)
        category = random.choice(categories)

        feedback_data.append((
            feedback_id, customer_name, customer_email, product_id,
            rating, review_text, feedback_date, ticket_status, category
        ))

    cursor.executemany('''
    INSERT OR REPLACE INTO customer_feedback VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', feedback_data)

    print(f"✓ Customer_feedback table created with {len(feedback_data)} records")

def create_inventory_2023_table(cursor):
    """Create and populate inventory_2023 table"""
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS inventory_2023 (
        inventory_id INTEGER PRIMARY KEY,
        product_id INTEGER,
        warehouse_location TEXT,
        quantity_in_stock INTEGER,
        reorder_level INTEGER,
        last_restock_date DATE,
        supplier_name TEXT,
        unit_cost REAL,
        FOREIGN KEY (product_id) REFERENCES products(product_id)
    )
    ''')

    warehouses = ['Warehouse A - NY', 'Warehouse B - CA', 'Warehouse C - TX',
                  'Warehouse D - FL', 'Warehouse E - IL']

    inventory_data = []

    for inventory_id in range(1, 201):
        product_id = random.randint(1, 20)
        warehouse = random.choice(warehouses)
        quantity = random.randint(0, 500)
        reorder_level = random.randint(20, 100)
        last_restock = fake.date_between(start_date='-6m', end_date='today')
        supplier = fake.company()
        unit_cost = round(random.uniform(10, 200), 2)

        inventory_data.append((
            inventory_id, product_id, warehouse, quantity, reorder_level,
            last_restock, supplier, unit_cost
        ))

    cursor.executemany('''
    INSERT OR REPLACE INTO inventory_2023 VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    ''', inventory_data)

    print(f"✓ Inventory_2023 table created with {len(inventory_data)} records")

def create_complete_database():
    """Create complete database with all tables and data"""
    print("\n" + "="*80)
    print("CREATING SAMPLE DATABASE: company.db")
    print("="*80 + "\n")

    conn, cursor = create_database()

    try:
        # Drop existing tables for fresh start
        tables = ['departments', 'employees', 'products', 'orders',
                  'sales_2023', 'customer_feedback', 'inventory_2023']
        for table in tables:
            cursor.execute(f"DROP TABLE IF EXISTS {table}")

        # Create all tables
        create_departments_table(cursor)
        create_employees_table(cursor)
        create_products_table(cursor)
        create_orders_table(cursor)
        create_sales_2023_table(cursor)
        create_customer_feedback_table(cursor)
        create_inventory_2023_table(cursor)

        conn.commit()

        print("\n" + "="*80)
        print("✓ DATABASE CREATED SUCCESSFULLY!")
        print("="*80 + "\n")

    except Exception as e:
        print(f"Error creating database: {e}")
        conn.rollback()
    finally:
        conn.close()

# Create the database
create_complete_database()


CREATING SAMPLE DATABASE: company.db

✓ Departments table created with 8 records
✓ Employees table created with 87 records
✓ Products table created with 20 records (with descriptions)


/tmp/ipykernel_2544/3624909865.py:94: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.executemany('''
/tmp/ipykernel_2544/3624909865.py:232: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.executemany('''
/tmp/ipykernel_2544/3624909865.py:279: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.executemany('''


✓ Orders table created with 500 records
✓ Sales_2023 table created with 2000 records
✓ Customer_feedback table created with 800 records
✓ Inventory_2023 table created with 200 records

✓ DATABASE CREATED SUCCESSFULLY!



/tmp/ipykernel_2544/3624909865.py:358: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.executemany('''
/tmp/ipykernel_2544/3624909865.py:399: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.executemany('''


In [ ]:
# ===============
# SECTION 2: INITIALIZE EMBEDDINGS MODEL
# ===============

print("\n" + "="*80)
print("INITIALIZING EMBEDDINGS MODEL")
print("="*80 + "\n")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

print("✓ HuggingFace embeddings model initialized")


INITIALIZING EMBEDDINGS MODEL



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ HuggingFace embeddings model initialized


In [ ]:
# ===============
# SECTION 3: CREATE TABLE DESCRIPTIONS VECTOR STORE (Scenario 1 Style)
# ===============

print("\n" + "="*80)
print("CREATING TABLE DESCRIPTIONS VECTOR STORE")
print("="*80 + "\n")

def get_table_descriptions():
    """Return detailed table descriptions for vector store"""
    return [
        {
            "table": "employees",
            "description": "Employee information including personal details, salary, department assignment, hire date, job position, and performance reviews. Use for queries about employee salaries, headcount, performance ratings, hiring trends, or department staffing."
        },
        {
            "table": "sales_2023",
            "description": "Sales transactions from 2023 including product sold, quantities, prices, discounts, and regional data. Use for queries about revenue, top selling products, sales by region, sales trends, or discount analysis."
        },
        {
            "table": "customer_feedback",
            "description": "Customer reviews, ratings, and support tickets. Contains detailed review text, star ratings, and feedback categories. Use for queries about customer satisfaction, product reviews, sentiment analysis, or support ticket status."
        },
        {
            "table": "products",
            "description": "Product catalog with names, categories, pricing, costs, SKUs, stock levels, and detailed product descriptions and features. Use for queries about product inventory, pricing, product features, or product categories."
        },
        {
            "table": "inventory_2023",
            "description": "Stock levels across different warehouse locations for 2023. Contains reorder levels, supplier information, and restocking dates. Use for queries about warehouse stock, suppliers, or inventory management."
        },
        {
            "table": "departments",
            "description": "Company department information including department names, managers, budgets, and office locations. Use for queries about department budgets, managers, or company structure."
        },
        {
            "table": "orders",
            "description": "Customer order records including order dates, shipping information, delivery status, payment methods, and total amounts. Use for queries about order status, shipping times, or payment analysis."
        },
    ]

# Create table descriptions vector store
table_descriptions = get_table_descriptions()
table_docs = [
    Document(
        page_content=f"Table: {t['table']}\nDescription: {t['description']}",
        metadata={"table_name": t['table']}
    )
    for t in table_descriptions
]

table_vectorstore = FAISS.from_documents(
    documents=table_docs,
    embedding=embeddings
)

print(f"✓ Table descriptions vector store created with {len(table_docs)} tables")



CREATING TABLE DESCRIPTIONS VECTOR STORE

✓ Table descriptions vector store created with 7 tables


In [ ]:
# ================
# SECTION 4: CREATE DATA CONTENT VECTOR STORE (Scenario 2 Style)
# ================

print("\n" + "="*80)
print("CREATING DATA CONTENT VECTOR STORE")
print("="*80 + "\n")

def create_data_vectorstore():
    """
    Create vector store for actual data content that benefits from semantic search:
    - Product descriptions and features
    - Customer reviews and feedback
    - Employee positions and skills (implied)
    """
    conn = sqlite3.connect("company.db")
    docs = []

    # 1. Product descriptions and features (most important for semantic search)
    print("  - Embedding product descriptions...")
    products = conn.execute("""
        SELECT product_id, product_name, category, price, description, features
        FROM products
    """).fetchall()

    for prod in products:
        doc = Document(
            page_content=f"Product: {prod[1]}\nCategory: {prod[2]}\nPrice: ${prod[3]}\nDescription: {prod[4]}\nFeatures: {prod[5]}",
            metadata={
                "source": "products",
                "product_id": prod[0],
                "product_name": prod[1],
                "category": prod[2],
                "price": prod[3]
            }
        )
        docs.append(doc)

    # 2. Customer feedback/reviews
    print("  - Embedding customer reviews...")
    feedback = conn.execute("""
        SELECT cf.feedback_id, cf.customer_name, cf.product_id, cf.rating,
               cf.review_text, cf.category, p.product_name
        FROM customer_feedback cf
        LEFT JOIN products p ON cf.product_id = p.product_id
    """).fetchall()

    for fb in feedback:
        product_name = fb[6] if fb[6] else f"Product #{fb[2]}"
        doc = Document(
            page_content=f"Customer Review for {product_name}:\nRating: {fb[3]}/5 stars\nReview: {fb[4]}\nCategory: {fb[5]}",
            metadata={
                "source": "customer_feedback",
                "feedback_id": fb[0],
                "product_id": fb[2],
                "product_name": product_name,
                "rating": fb[3],
                "feedback_category": fb[5]
            }
        )
        docs.append(doc)

    # 3. Employee positions (useful for "find people who..." queries)
    print("  - Embedding employee information...")
    employees = conn.execute("""
        SELECT e.employee_id, e.first_name, e.last_name, e.position,
               e.performance_rating, d.department_name
        FROM employees e
        LEFT JOIN departments d ON e.department_id = d.department_id
    """).fetchall()

    for emp in employees:
        doc = Document(
            page_content=f"Employee: {emp[1]} {emp[2]}\nPosition: {emp[3]}\nDepartment: {emp[5]}\nPerformance Rating: {emp[4]}/5",
            metadata={
                "source": "employees",
                "employee_id": emp[0],
                "full_name": f"{emp[1]} {emp[2]}",
                "position": emp[3],
                "department": emp[5]
            }
        )
        docs.append(doc)

    conn.close()

    vectorstore = FAISS.from_documents(documents=docs, embedding=embeddings)
    print(f"✓ Data content vector store created with {len(docs)} documents")

    return vectorstore

data_vectorstore = create_data_vectorstore()


CREATING DATA CONTENT VECTOR STORE

  - Embedding product descriptions...
  - Embedding customer reviews...
  - Embedding employee information...
✓ Data content vector store created with 907 documents


In [ ]:
# =================
# SECTION 5: INITIALIZE LLM AND DATABASE
# =================

print("\n" + "="*80)
print("INITIALIZING LLM AND DATABASE CONNECTION")
print("="*80 + "\n")

# Initialize Groq LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",  # Fast model with good rate limits
    temperature=0,
    max_tokens=2048,
    max_retries=2,
)

print("✓ Groq LLM initialized (llama-3.1-8b-instant)")

# Full database connection
db_full = SQLDatabase.from_uri(
    "sqlite:///company.db",
    sample_rows_in_table_info=2
)

print("✓ Database connection established")
print(f"  Available tables: {db_full.get_usable_table_names()}")


INITIALIZING LLM AND DATABASE CONNECTION

✓ Groq LLM initialized (llama-3.1-8b-instant)
✓ Database connection established
  Available tables: ['customer_feedback', 'departments', 'employees', 'inventory_2023', 'orders', 'products', 'sales_2023']


Note: We store table as embeddings and we store data inside the table also as embeddings. We don't store in the same vectorstore. That will give problem during retriving the data.
That's why here we create two different vectorstore
1. table_vectorstore
2. vectorstore


Now we write the function to get data using functions. For that relevant vectorstore we will use.
- Means for finding tables we use table_vectorstore and for  
- similarity search based on data inside the table we use vectorstore

In [ ]:
# =================
# SECTION 6: CORE HELPER FUNCTIONS
# =================

def find_relevant_tables(question: str, k: int = 3) -> List[str]:
    """
    Find relevant tables using table description embeddings (Scenario 1 approach).
    Used when we need to narrow down which tables to query.
    """
    results = table_vectorstore.similarity_search(question, k=k)

    # "results" is array of Document() object.

    return [doc.metadata['table_name'] for doc in results]

def semantic_data_search(query: str, k: int = 5) -> str:
    """
    Search actual data content semantically (Scenario 2 approach).
    Used for feature-based, description-based, or sentiment-based queries.
    """
    results = data_vectorstore.similarity_search(query, k=k)

    output = []
    for i, doc in enumerate(results, 1):
        # enumerate(results, 1) : Give each document a number starting from 1 (not 0).
        # i = 1,2,3,...
        # doc = actual Document() object

        source = doc.metadata.get('source', 'unknown') # While creating document we set the value for "source" equal to name of the table. It's not the link
        output.append(f"[Result {i} - {source}]\n{doc.page_content}")

    return "\n\n" + "-"*40 + "\n\n".join(output) if output else "No relevant data found."

def classify_query(query: str) -> str:
    """
    Classify query type to determine which approach to use.
    Returns: 'SQL', 'SEMANTIC', or 'HYBRID'
    """
    query_lower = query.lower()

    # SQL-only indicators (numerical, aggregations, exact filtering)
    sql_keywords = [
        'average', 'avg', 'count', 'how many', 'total', 'sum', 'max', 'min',
        'list all', 'show all', 'get all', 'between', 'greater than', 'less than',
        'highest', 'lowest', 'top', 'bottom', 'number of', 'calculate',
        'group by', 'order by', 'sort by', 'compare', 'per department',
        'per region', 'by category', 'over $', 'under $', 'more than', 'fewer than'
    ]

    # Semantic indicators (natural language, features, opinions)
    semantic_keywords = [
        'recommend', 'suggest', 'best for', 'suitable for', 'good for',
        'perfect for', 'ideal for', 'designed for', 'meant for',
        'like', 'similar to', 'reviews', 'feedback', 'opinion', 'sentiment',
        'what do customers say', 'think about', 'feel about',
        'describe', 'features', 'characteristics', 'qualities',
        'find something', 'looking for', 'i need', 'help me find',
        'who can', 'who has experience', 'skilled in', 'specialized in'
    ]

    has_sql = any(kw in query_lower for kw in sql_keywords)

    # It returns one boolean value at a time (True or False), and any() checks them one by one.
    # "kw in query_lower for kw in sql_keywords" This is a generator expression (not a list).

    # the generator expression inside the any() function
    # It works like this internally:
        # 1. It starts looping through sql_keywords one by one.
        # 2. For each keyword (kw), it checks: kw in query_lower → This gives True or False.
        # 3. It immediately sends that True/False to the any() function.
        # 4. any() says: "If I get even one True, I will return True right away and stop checking further."
    # It does not create a full list like [False, False, True, False, ...] in memory.

    # Think of any() as a security guard at the door. The generator is like people coming one by one. As soon as it see first True. stop checking others.


    # any([kw in query_lower for kw in sql_keywords])   # Note the []
    # This would create a full boolean list, and then passs that list to any()

    # like list comprehension we create a tuple comprehension then it will become a generator

    # (kw in query_lower for kw in sql_keywords)
    # It does not create a tuple. It creates a generator object.
    # To actually create tuple, You have to use tuple() constructor
    # tuple(kw in query_lower for kw in sql_keywords)   # This creates a real tuple


    has_semantic = any(kw in query_lower for kw in semantic_keywords)

    if has_sql and has_semantic:
        return "HYBRID"
    elif has_semantic:
        return "SEMANTIC"
    else:
        return "SQL"

In [ ]:
# ================
# SECTION 7: CREATE TOOLS FOR AGENT
# ================

# Semantic search tool
semantic_tool = Tool(
    name="semantic_data_search",
    func=semantic_data_search,
    description="""Use this tool to find information based on features, descriptions, reviews,
    or natural language characteristics. Good for queries like:
    - 'Find products for gaming/fitness/outdoor use'
    - 'What do customers say about product quality?'
    - 'Find employees with experience in engineering'
    - 'Reviews mentioning delivery problems'
    - 'Products suitable for beginners'
    Do NOT use for numerical queries like averages, counts, or totals."""
)

# Table finder tool
def table_finder_func(query: str) -> str:
    tables = find_relevant_tables(query, k=3)
    return f"Most relevant tables for '{query}': {', '.join(tables)}"

table_finder_tool = Tool(
    name="find_relevant_tables",
    func=table_finder_func,
    description="Find which database tables are most relevant for a given question. Use this first for complex queries."
)

In [ ]:
# ============================================================================
# SECTION 8: MAIN FUSION QUERY FUNCTION
# ============================================================================

def fusion_query(user_question: str, verbose: bool = True) -> str:
    """
    MAIN FUSION FUNCTION: Intelligently routes queries to the appropriate method.

    Workflow:
    1. Classify the query type (SQL, SEMANTIC, or HYBRID)
    2. For SQL: Find relevant tables → Create focused SQL agent → Execute
    3. For SEMANTIC: Use data vector store for semantic search → Generate response
    4. For HYBRID: Combine both approaches for comprehensive answer

    Args:
        user_question: Natural language question from user
        verbose: Whether to print detailed progress

    Returns:
        String response to the user's question
    """

    if verbose:
        print(f"\n{'='*80}")
        print(f"🔍 QUESTION: {user_question}")
        print(f"{'='*80}\n")

    # Step 1: Classify query type
    query_type = classify_query(user_question)
    if verbose:
        print(f"📋 Query Classification: {query_type}\n")

    # Step 2: Execute based on type

    # -------------------------------------------------------------------------
    # SEMANTIC ONLY: Use vector search for feature/description queries
    # -------------------------------------------------------------------------
    if query_type == "SEMANTIC":
        if verbose:
            print("🔎 Using SEMANTIC SEARCH (no SQL needed)...\n")

        # Get relevant documents
        search_results = semantic_data_search(user_question, k=8)

        # Generate natural language response
        response_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a helpful assistant analyzing search results from a company database.
Based on the search results provided, give a clear, well-organized answer to the user's question.
If the results include reviews, summarize the sentiment.
If the results include products, highlight key features.
Format your response nicely with bullet points where appropriate."""),
            ("human", "Question: {question}\n\nSearch Results:\n{results}\n\nProvide a helpful, well-formatted answer:")
        ])

        chain = response_prompt | llm | StrOutputParser()
        response = chain.invoke({"question": user_question, "results": search_results})

        if verbose:
            print(f"\n{'='*80}")
            print(f"✅ ANSWER:")
            print(f"{'='*80}")

        return response

    # -------------------------------------------------------------------------
    # SQL ONLY: Use SQL agent for numerical/aggregate queries
    # -------------------------------------------------------------------------
    elif query_type == "SQL":
        if verbose:
            print("🗄️ Using SQL AGENT...\n")

        # Find relevant tables (Scenario 1 approach)
        relevant_tables = find_relevant_tables(user_question, k=4)
        if verbose:
            print(f"📊 Relevant tables identified: {relevant_tables}\n")

        # Create database connection with only relevant tables
        db_filtered = SQLDatabase.from_uri(
            "sqlite:///company.db",
            include_tables=relevant_tables,
            sample_rows_in_table_info=2
        )

        # Create SQL toolkit and agent
        toolkit = SQLDatabaseToolkit(db=db_filtered, llm=llm)
        sql_tools = toolkit.get_tools()

        # Use initialize_agent for compatibility with Groq
        agent = initialize_agent(
            tools=sql_tools,
            llm=llm,
            agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
            verbose=verbose,
            handle_parsing_errors=True,
            max_iterations=10,
            early_stopping_method="generate"
        )

        result = agent.invoke({"input": user_question})

        if verbose:
            print(f"\n{'='*80}")
            print(f"✅ ANSWER:")
            print(f"{'='*80}")

        return result['output']

    # -------------------------------------------------------------------------
    # HYBRID: Combine SQL and Semantic for complex queries
    # -------------------------------------------------------------------------
    else:  # HYBRID
        if verbose:
            print("🔀 Using HYBRID APPROACH (SQL + Semantic)...\n")

        context_parts = []

        # Part 1: Get semantic context
        if verbose:
            print("  Step 1: Running semantic search...")
        semantic_results = semantic_data_search(user_question, k=5)
        context_parts.append(f"SEMANTIC SEARCH RESULTS:\n{semantic_results}")

        # Part 2: Get SQL results
        if verbose:
            print("  Step 2: Running SQL query...")

        relevant_tables = find_relevant_tables(user_question, k=4)
        if verbose:
            print(f"  Relevant tables: {relevant_tables}")

        db_filtered = SQLDatabase.from_uri(
            "sqlite:///company.db",
            include_tables=relevant_tables,
            sample_rows_in_table_info=2
        )

        # Generate SQL query
        sql_prompt = ChatPromptTemplate.from_messages([
            ("system", f"""Generate a SQLite query for the given question.
Database schema:
{db_filtered.get_table_info()}

Return ONLY the SQL query, no explanation. If the question cannot be answered with SQL alone, return 'NO_SQL_NEEDED'."""),
            ("human", "{query}")
        ])

        sql_generator = sql_prompt | llm | StrOutputParser()
        sql_query = sql_generator.invoke({"query": user_question})
        sql_query = sql_query.replace("```sql", "").replace("```", "").strip()

        if sql_query != "NO_SQL_NEEDED":
            try:
                if verbose:
                    print(f"  Generated SQL: {sql_query[:100]}...")
                sql_results = db_filtered.run(sql_query)
                context_parts.append(f"SQL QUERY RESULTS:\nQuery: {sql_query}\nResults: {sql_results}")
            except Exception as e:
                context_parts.append(f"SQL Error (falling back to semantic only): {e}")

        # Combine and generate response
        combined_context = "\n\n" + "="*40 + "\n\n".join(context_parts)

        response_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a helpful assistant with access to both database queries and semantic search results.
Synthesize the information from both sources to provide a comprehensive, well-organized answer.
Use specific data points from SQL results when available.
Include qualitative insights from semantic search (like reviews or descriptions).
Format your response clearly with sections if needed."""),
            ("human", "Question: {question}\n\nAvailable Information:{context}\n\nProvide a comprehensive answer:")
        ])

        chain = response_prompt | llm | StrOutputParser()
        response = chain.invoke({"question": user_question, "context": combined_context})

        if verbose:
            print(f"\n{'='*80}")
            print(f"✅ ANSWER:")
            print(f"{'='*80}")

        return response

In [ ]:
# ============================================================================
# SECTION 9: SIMPLE ALTERNATIVE (No Agent Loops)
# ============================================================================

def simple_fusion_query(user_question: str) -> str:
    """
    Simpler version without complex agent loops - more reliable for quick queries.
    Uses direct SQL generation and semantic search without ReAct loops.
    """

    print(f"\n{'='*80}")
    print(f"🔍 QUESTION: {user_question}")
    print(f"{'='*80}\n")

    query_type = classify_query(user_question)
    print(f"📋 Query Classification: {query_type}\n")

    context = ""

    # Get semantic context if needed
    if query_type in ["SEMANTIC", "HYBRID"]:
        print("🔎 Fetching semantic search results...")
        semantic_results = semantic_data_search(user_question, k=5)
        context += f"\n\nSEMANTIC SEARCH RESULTS:\n{semantic_results}"

    # Get SQL results if needed
    if query_type in ["SQL", "HYBRID"]:
        print("🗄️ Generating and executing SQL...")

        relevant_tables = find_relevant_tables(user_question, k=3)
        print(f"   Relevant tables: {relevant_tables}")

        db_filtered = SQLDatabase.from_uri(
            "sqlite:///company.db",
            include_tables=relevant_tables,
            sample_rows_in_table_info=2
        )

        # Generate SQL
        sql_prompt = ChatPromptTemplate.from_messages([
            ("system", f"""You are a SQL expert. Generate a SQLite query for this question.
Database schema:
{db_filtered.get_table_info()}

Return ONLY the SQL query. No explanation, no markdown."""),
            ("human", "{query}")
        ])

        sql_generator = sql_prompt | llm | StrOutputParser()
        sql_query = sql_generator.invoke({"query": user_question})
        sql_query = sql_query.replace("```sql", "").replace("```", "").replace("```", "").strip()

        print(f"   Generated SQL: {sql_query[:80]}...")

        try:
            sql_results = db_filtered.run(sql_query)
            context += f"\n\nSQL QUERY: {sql_query}\nSQL RESULTS: {sql_results}"
        except Exception as e:
            print(f"   SQL Error: {e}")
            context += f"\n\nSQL Error: {e}"

    # Generate final response
    response_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a helpful database assistant. Based on the context provided,
give a clear, well-formatted answer to the user's question.
- Use specific numbers and data when available
- Highlight key insights
- Format with bullet points or sections for clarity"""),
        ("human", "Question: {question}\n\nContext:{context}\n\nProvide a helpful answer:")
    ])

    chain = response_prompt | llm | StrOutputParser()
    response = chain.invoke({"question": user_question, "context": context})

    print(f"\n{'='*80}")
    print(f"✅ ANSWER:")
    print(f"{'='*80}\n")

    return response

In [ ]:
# ============================================================================
# SECTION 10: TEST THE FUSION APPROACH
# ============================================================================

print("\n" + "="*80)
print("🧪 TESTING THE FUSION APPROACH")
print("="*80)

# Test 1: Pure SQL Query (numerical/aggregate)
print("\n\n" + "🔵"*40)
print("TEST 1: SQL Query (Average Calculation)")
print("🔵"*40)
result1 = simple_fusion_query("What is the average salary of employees in each department?")
print(result1)



🧪 TESTING THE FUSION APPROACH


🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵
TEST 1: SQL Query (Average Calculation)
🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵

🔍 QUESTION: What is the average salary of employees in each department?

📋 Query Classification: SQL

🗄️ Generating and executing SQL...
   Relevant tables: ['employees', 'departments', 'sales_2023']
   Generated SQL: SELECT department_id, AVG(salary) FROM employees GROUP BY department_id...

✅ ANSWER:

**Average Salary by Department**

Based on the provided SQL results, here's a breakdown of the average salary for each department:

### Department 1
* Average Salary: $92,539.80
* Note: This department has the highest average salary among all departments.

### Department 2
* Average Salary: $82,451.53
* Note: This department has the lowest average salary among all departments.

### Department 3
* Average Salary: $101,442.99
* Note: This department has a significantly higher average salary compared to the overall average.

### Departm

In [ ]:

# Test 2: Pure Semantic Query (feature-based search)
print("\n\n" + "🟢"*40)
print("TEST 2: Semantic Query (Product Features)")
print("🟢"*40)
result2 = simple_fusion_query("I need a product suitable for outdoor activities and adventures")
print(result2)






🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢
TEST 2: Semantic Query (Product Features)
🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢

🔍 QUESTION: I need a product suitable for outdoor activities and adventures

📋 Query Classification: SEMANTIC

🔎 Fetching semantic search results...

✅ ANSWER:

**Recommended Products for Outdoor Activities and Adventures**

Based on the provided semantic search results, we have identified three products that are suitable for outdoor activities and adventures:

### 1. **Mountain Bike Explorer**

* **Category:** Sports
* **Price:** $899.99
* **Description:** All-terrain mountain bike for adventure seekers and trail riders. Handles rough terrain with ease.
* **Features:**
	+ 21-speed
	+ Hydraulic brakes
	+ Front suspension
	+ Aluminum frame

**Customer Reviews:**

* 4/5 stars: "Five stars! The product is durable, well-designed, and easy to use. Will definitely buy again."
* 5/5 stars: "This product has changed my daily routine for the better. Couldn't be happier 

In [ ]:
# Test 3: Pure Semantic Query (customer reviews)
print("\n\n" + "🟢"*40)
print("TEST 3: Semantic Query (Customer Sentiment)")
print("🟢"*40)
result3 = simple_fusion_query("What do customers say about product quality? Are they satisfied?")
print(result3)





🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢
TEST 3: Semantic Query (Customer Sentiment)
🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢

🔍 QUESTION: What do customers say about product quality? Are they satisfied?

📋 Query Classification: SEMANTIC

🔎 Fetching semantic search results...

✅ ANSWER:

**Customer Satisfaction with Product Quality: Key Insights**

Based on the provided semantic search results, we can analyze customer feedback on product quality. Here are the key findings:

**Overall Rating:**
- The average rating for product quality is 3.6/5 stars (calculated from 5 reviews).
- However, there are two 5-star reviews that significantly impact the average rating.

**Customer Feedback:**

* **Positive Feedback (2 reviews):**
	+ Review 3: Customer is "really satisfied" with the purchase and praises the product's performance and packaging.
	+ Review 4: Customer is "impressed" with the quality and delivery speed of the Mountain Bike Explorer.
* **Negative Feedback (3 reviews):**
	+ Review 

In [ ]:
# Test 4: Hybrid Query
print("\n\n" + "🟡"*40)
print("TEST 4: Hybrid Query (Recommend + Data)")
print("🟡"*40)
result4 = simple_fusion_query("Recommend top-selling products and show what customers think about them")
print(result4)




🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡
TEST 4: Hybrid Query (Recommend + Data)
🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡

🔍 QUESTION: Recommend top-selling products and show what customers think about them

📋 Query Classification: HYBRID

🔎 Fetching semantic search results...
🗄️ Generating and executing SQL...
   Relevant tables: ['products', 'customer_feedback', 'sales_2023']
   Generated SQL: SELECT p.product_name, SUM(s.quantity) AS total_quantity, AVG(cf.rating) AS aver...

✅ ANSWER:

**Top-Selling Products and Customer Feedback**

Based on the provided SQL query results, the top 5 selling products in 2023 are:

1. **Fitness Tracker Band**: 25,632 units sold
	* Average customer rating: 4.04/5 stars
2. **Wireless Headphones Pro**: 25,370 units sold
	* Average customer rating: 3.84/5 stars
3. **Smart LED Bulbs Set**: 25,008 units sold
	* Average customer rating: 3.73/5 stars
4. **Mountain Bike Explorer**: 24,804 units sold
	* Average customer rating: 4.10/5 stars
5. **USB-C Hub Pr

In [ ]:

# Test 5: SQL Query with agent (more complex)
print("\n\n" + "🔵"*40)
print("TEST 5: SQL Query with Agent (Sales Analysis)")
print("🔵"*40)
result5 = fusion_query("What were the total sales in 2023 grouped by region?")
print(result5)





🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵
TEST 5: SQL Query with Agent (Sales Analysis)
🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵

🔍 QUESTION: What were the total sales in 2023 grouped by region?

📋 Query Classification: SQL

🗄️ Using SQL AGENT...

📊 Relevant tables identified: ['sales_2023', 'inventory_2023', 'products', 'departments']



> Entering new AgentExecutor chain...


/tmp/ipykernel_2544/1629699910.py:89: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the [LangGraph documentation](https://langchain-ai.github.io/langgraph/) as well as guides for [Migrating from AgentExecutor](https://python.langchain.com/docs/how_to/migrate_agent/) and LangGraph's [Pre-built ReAct agent](https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/).
  agent = initialize_agent(


Thought: To answer this question, I need to know the structure of the tables involved and the fields that contain the sales data and region information.
Action: sql_db_schema
Action Input: sales, regions
Observation: Error: table_names {'regions', 'sales'} not found in database
Thought:Thought: It seems that the tables 'sales' and 'regions' do not exist in the database. I need to find the actual table names that contain the sales data and region information.

Action: sql_db_list_tables
Action Input: 
Observation: departments, inventory_2023, products, sales_2023
Thought:Question: What were the total sales in 2023 grouped by region?
Thought: I now know the actual table names that contain the sales data and region information.
Action: sql_db_schema
Action Input: sales_2023, regions
Observation: Error: table_names {'regions'} not found in database
Thought:Question: What were the total sales in 2023 grouped by region?
Thought: It seems that the table 'regions' does not exist in the databas

KeyboardInterrupt: 

In [ ]:
# Test 6: Finding specific employees
print("\n\n" + "🟢"*40)
print("TEST 6: Semantic Query (Employee Search)")
print("🟢"*40)
result6 = simple_fusion_query("Find employees who work in engineering or software development")
print(result6)



🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢
TEST 6: Semantic Query (Employee Search)
🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢

🔍 QUESTION: Find employees who work in engineering or software development

📋 Query Classification: SQL

🗄️ Generating and executing SQL...
   Relevant tables: ['employees', 'departments', 'inventory_2023']
   Generated SQL: SELECT * FROM employees WHERE position IN ('Software Engineer', 'Engineering Man...

✅ ANSWER:

**Employees Working in Engineering or Software Development**

Based on the provided SQL query results, we can identify the employees who work in engineering or software development. Here are the results:

### Employees in Software Development

* **Employee ID**: 25, 32, 34, 36
* **Name**: Nancy Edwards, Robert Brown, Michelle Smith, Danielle Rodriguez
* **Email**: nancy.edwards25@company.com, robert.brown32@company.com, michelle.smith34@company.com, danielle.rodriguez36@company.com
* **Department ID**: 3
* **Position**: Software Engineer
* **Salar

In [ ]:
# ============================================================================
# SECTION 11: INTERACTIVE QUERY FUNCTION
# ============================================================================

def ask(question: str, use_agent: bool = False) -> str:
    """
    Convenient function to ask questions.

    Args:
        question: Your question in natural language
        use_agent: If True, uses the agent-based approach (more powerful but slower)
                   If False, uses the simple approach (faster and more reliable)

    Example:
        ask("What is the average product price?")
        ask("Find products good for fitness", use_agent=True)
    """
    if use_agent:
        return fusion_query(question)
    else:
        return simple_fusion_query(question)

In [ ]:
# ============================================================================
# USAGE EXAMPLES
# ============================================================================

print("\n" + "="*80)
print("📚 USAGE EXAMPLES")
print("="*80)
print("""
# SQL Queries (aggregations, counting, filtering):
ask("How many employees are in each department?")
ask("What is the total revenue from sales in 2023?")
ask("Show the top 5 highest paid employees")
ask("List products with price greater than $200")

# Semantic Queries (features, descriptions, reviews):
ask("Find products perfect for home workouts")
ask("What are customers saying about our products?")
ask("Find employees with experience in marketing")
ask("Recommend products suitable for beginners")

# Hybrid Queries (combines both):
ask("Which products have the best reviews and highest sales?")
ask("Recommend budget-friendly products that customers love")

# Using the agent (for complex queries):
ask("Calculate average employee performance by department and compare with budget", use_agent=True)
""")

print("\n✅ Fusion approach is ready! Use ask('your question') to query the database.")


📚 USAGE EXAMPLES

# SQL Queries (aggregations, counting, filtering):
ask("How many employees are in each department?")
ask("What is the total revenue from sales in 2023?")
ask("Show the top 5 highest paid employees")
ask("List products with price greater than $200")

# Semantic Queries (features, descriptions, reviews):
ask("Find products perfect for home workouts")
ask("What are customers saying about our products?")
ask("Find employees with experience in marketing")
ask("Recommend products suitable for beginners")

# Hybrid Queries (combines both):
ask("Which products have the best reviews and highest sales?")
ask("Recommend budget-friendly products that customers love")

# Using the agent (for complex queries):
ask("Calculate average employee performance by department and compare with budget", use_agent=True)


✅ Fusion approach is ready! Use ask('your question') to query the database.


```text
┌─────────────────────────────────────────────────────────────────┐
│                        USER QUESTION                            │
└─────────────────────────┬───────────────────────────────────────┘
                          │
                          ▼
              ┌───────────────────────┐
              │   QUERY CLASSIFIER    │
              │  (Keyword Analysis)   │
              └───────────┬───────────┘
                          │
          ┌───────────────┼───────────────┐
          │               │               │
          ▼               ▼               ▼
    ┌─────────┐     ┌─────────┐     ┌─────────┐
    │   SQL   │     │ SEMANTIC│     │ HYBRID  │
    └────┬────┘     └────┬────┘     └────┬────┘
         │               │               │
         ▼               ▼               ▼
┌────────────────┐ ┌──────────────┐ ┌────────────────┐
│ Find Relevant  │ │ Vector Search│ │ Both Approaches│
│ Tables (FAISS) │ │ (Data Store) │ │   Combined     │
├────────────────┤ ├──────────────┤ ├────────────────┤
│ Create Focused │ │ Get Similar  │ │ SQL Results +  │
│ SQL Agent      │ │ Documents    │ │ Semantic Hits  │
├────────────────┤ ├──────────────┤ ├────────────────┤
│ Execute Query  │ │ LLM Response │ │ Synthesized    │
│ Return Results │ │              │ │ Answer         │
└────────────────┘ └──────────────┘ └────────────────┘
```